In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# Find the repository root
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "Notebooks 2").exists()
        and (PROJECT_ROOT / "app").exists()
    ):
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT ROOT:", PROJECT_ROOT)

paths_to_check = [
    "data/raw",
    "data/interim/patients.csv",
    "data/interim/recordings.csv",
    "data/interim/cycles.csv",
    "data/processed/features/respiratory_cycle_features.csv",
    "data/processed/deep_learning/train_manifest.csv",
    "data/processed/deep_learning/test_manifest.csv",
    "data/processed/classical_ml/classical_ml_selected_features.csv",
]

print("\n=== DATA AVAILABILITY ===")

for relative_path in paths_to_check:
    path = PROJECT_ROOT / relative_path
    print(f"{'EXISTS' if path.exists() else 'MISSING':7} | {relative_path}")

print("\n=== RAW DATASET FOLDERS ===")

raw_dir = PROJECT_ROOT / "data" / "raw"

if raw_dir.exists():
    for item in sorted(raw_dir.iterdir()):
        if item.is_dir():
            wav_count = sum(1 for _ in item.rglob("*.wav"))
            print(f"{item.name}: {wav_count} WAV files")
        else:
            print(f"{item.name}: {item.stat().st_size / 1024:.1f} KB")
else:
    print("Raw data directory not found.")

print("\n=== AVAILABLE CSV METADATA ===")

for path in sorted((PROJECT_ROOT / "data").rglob("*.csv")):
    try:
        df = pd.read_csv(path, nrows=3)
        print(f"\n{path.relative_to(PROJECT_ROOT)}")
        print("Columns:", df.columns.tolist())
    except Exception as exc:
        print(f"\nCould not read {path.name}: {exc}")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "Notebooks 2").exists()
        and (PROJECT_ROOT / "app").exists()
    ):
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE = (
    PROJECT_ROOT
    / "data/processed/deep_learning/"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

files_to_inspect = [
    BASE / "experiment_b_class_support_audit.csv",
    BASE / "experiment_b_feature_cache/experiment_b_v1_strict/evaluation/experiment_b_test_summary.csv",
    BASE / "experiment_c1_icbhi_only/evaluation/c1_test_summary.csv",
    BASE / "experiment_c2_icbhi_fraiwan_high_capacity/c2_train_manifest.csv",
    PROJECT_ROOT / "data/processed/deep_learning/train_manifest.csv",
    PROJECT_ROOT / "data/processed/deep_learning/test_manifest.csv",
]

for path in files_to_inspect:
    print("\n" + "=" * 80)
    print(path.relative_to(PROJECT_ROOT))
    print("Exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    display(df.head(12))

    if "diagnosis" in df.columns:
        print("\nDiagnosis counts:")
        display(df["diagnosis"].value_counts(dropna=False).to_frame("rows"))

    if {"patient_id", "diagnosis"}.issubset(df.columns):
        print("\nUnique patients by diagnosis:")
        display(
            df.groupby("diagnosis")["patient_id"]
            .nunique()
            .sort_values(ascending=False)
            .to_frame("patients")
        )

print("\nAudit complete.")

In [ ]:

from pathlib import Path
import pandas as pd

BASE = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"

# Inspect the existing split manifests without modifying any files.
manifest_paths = [
    ("ICBHI original train", PROJECT_ROOT / "data/processed/deep_learning/train_manifest.csv"),
    ("ICBHI original test", PROJECT_ROOT / "data/processed/deep_learning/test_manifest.csv"),
    ("Experiment B train", BASE / "experiment_b_train_manifest.csv"),
    ("Experiment B ICBHI test", BASE / "experiment_b_icbhi_test_manifest.csv"),
    ("Experiment B Fraiwan test", BASE / "experiment_b_fraiwan_test_manifest.csv"),
]

# Also discover C2 split manifests, without assuming their exact filenames.
c2_dir = BASE / "experiment_c2_icbhi_fraiwan_high_capacity"
for path in sorted(c2_dir.glob("*manifest*.csv")):
    manifest_paths.append((f"Experiment C2: {path.name}", path))

loaded = {}

for name, path in manifest_paths:
    print("\n" + "=" * 85)
    print(name)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    loaded[name] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    patient_col = next(
        (c for c in ["patient_id", "patient", "subject_id", "subject", "patientID"]
         if c in df.columns),
        None
    )
    recording_col = next(
        (c for c in ["recording_id", "source_recording_id", "recording",
                     "recording_path", "audio_path"]
         if c in df.columns),
        None
    )
    split_col = next(
        (c for c in ["split", "official_split", "partition"]
         if c in df.columns),
        None
    )

    if patient_col:
        print("Patient column:", patient_col)
        print("Unique patients:", df[patient_col].nunique())
        if split_col:
            print("Patients by split:")
            print(df.groupby(split_col)[patient_col].nunique().to_string())

    if recording_col:
        print("Recording column:", recording_col)
        print("Unique recording values:", df[recording_col].nunique())
        print("Duplicate recording rows:", int(df[recording_col].duplicated().sum()))

    if "diagnosis" in df.columns:
        print("Recording/row counts by diagnosis:")
        print(df["diagnosis"].value_counts(dropna=False).to_string())

# 1. Check patient overlap in the original ICBHI manifests.
print("\n" + "=" * 85)
print("PATIENT OVERLAP CHECKS")

train = loaded.get("ICBHI original train")
test = loaded.get("ICBHI original test")

if train is not None and test is not None and "patient_id" in train.columns and "patient_id" in test.columns:
    train_patients = set(train["patient_id"].dropna().astype(str))
    test_patients = set(test["patient_id"].dropna().astype(str))
    overlap = sorted(train_patients & test_patients)
    print("Original ICBHI train/test patient overlap:", len(overlap))
    print("Overlapping patient IDs:", overlap)
else:
    print("Could not perform original ICBHI patient overlap check: patient_id column missing.")

# 2. Check whether the combined Experiment B training patients occur in either test set.
b_train = loaded.get("Experiment B train")
b_icbhi_test = loaded.get("Experiment B ICBHI test")
b_fraiwan_test = loaded.get("Experiment B Fraiwan test")

def find_patient_col(df):
    return next(
        (c for c in ["patient_id", "patient", "subject_id", "subject", "patientID"]
         if c in df.columns),
        None
    )

if b_train is not None:
    train_pcol = find_patient_col(b_train)
    if train_pcol:
        b_train_patients = set(b_train[train_pcol].dropna().astype(str))

        for test_name, test_df in [
            ("Experiment B ICBHI test", b_icbhi_test),
            ("Experiment B Fraiwan test", b_fraiwan_test),
        ]:
            if test_df is None:
                continue
            test_pcol = find_patient_col(test_df)
            if test_pcol:
                test_patients = set(test_df[test_pcol].dropna().astype(str))
                overlap = sorted(b_train_patients & test_patients)
                print(f"{test_name}: train/test patient overlap = {len(overlap)}")
                if overlap:
                    print("Overlapping IDs:", overlap)
            else:
                print(f"{test_name}: no recognizable patient ID column.")
    else:
        print("Experiment B train: no recognizable patient ID column.")

# 3. Print the discovered C2 manifests and their split labels.
print("\n" + "=" * 85)
print("C2 MANIFEST SPLIT SUMMARY")

c2_found = False
for name, df in loaded.items():
    if name.startswith("Experiment C2"):
        c2_found = True
        print("\n", name)
        split_col = next(
            (c for c in ["split", "official_split", "partition"] if c in df.columns),
            None
        )
        if split_col:
            print(df[split_col].value_counts(dropna=False).to_string())
        else:
            print("No split column found; inspect the columns and manifest meaning above.")

if not c2_found:
    print("No C2 manifest files matched '*manifest*.csv'.")

print("\nAudit only complete. No models or files were changed.")


In [ ]:

import pandas as pd
from pathlib import Path

BASE = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
C2 = BASE / "experiment_c2_icbhi_fraiwan_high_capacity"

split_files = {
    "B_train": BASE / "experiment_b_train_manifest.csv",
    "B_ICBHI_test": BASE / "experiment_b_icbhi_test_manifest.csv",
    "B_Fraiwan_test": BASE / "experiment_b_fraiwan_test_manifest.csv",
    "C2_train": C2 / "c2_train_manifest.csv",
    "C2_validation": C2 / "c2_validation_manifest.csv",
    "C2_ICBHI_test": C2 / "c2_icbhi_test_manifest.csv",
    "C2_Fraiwan_test": C2 / "c2_fraiwan_test_manifest.csv",
}

manifests = {}
for name, path in split_files.items():
    if path.exists():
        manifests[name] = pd.read_csv(path)
        print(f"{name}: {manifests[name].shape}")
    else:
        print(f"{name}: FILE MISSING -> {path}")

def patient_set(df):
    if df is None or "patient_id" not in df.columns:
        return set()
    return set(
        (df["source_dataset"].astype(str) + "::" +
         df["patient_id"].astype(str)).dropna()
    )

def recording_set(df):
    if df is None or not {"source_dataset", "source_recording_id"}.issubset(df.columns):
        return set()
    return set(
        (df["source_dataset"].astype(str) + "::" +
         df["source_recording_id"].astype(str)).dropna()
    )

print("\n" + "=" * 80)
print("1. PATIENT OVERLAP")

for train_name, test_names in [
    ("B_train", ["B_ICBHI_test", "B_Fraiwan_test"]),
    ("C2_train", ["C2_validation", "C2_ICBHI_test", "C2_Fraiwan_test"]),
    ("C2_validation", ["C2_ICBHI_test", "C2_Fraiwan_test"]),
]:
    if train_name not in manifests:
        continue
    for test_name in test_names:
        if test_name not in manifests:
            continue
        overlap = patient_set(manifests[train_name]) & patient_set(manifests[test_name])
        print(f"{train_name} vs {test_name}: {len(overlap)} shared dataset-patient IDs")
        if overlap:
            print("  Examples:", sorted(overlap)[:10])

print("\n" + "=" * 80)
print("2. SOURCE RECORDING OVERLAP")

for left, right in [
    ("B_train", "B_ICBHI_test"),
    ("B_train", "B_Fraiwan_test"),
    ("C2_train", "C2_validation"),
    ("C2_train", "C2_ICBHI_test"),
    ("C2_train", "C2_Fraiwan_test"),
    ("C2_validation", "C2_ICBHI_test"),
    ("C2_validation", "C2_Fraiwan_test"),
]:
    if left in manifests and right in manifests:
        overlap = recording_set(manifests[left]) & recording_set(manifests[right])
        print(f"{left} vs {right}: {len(overlap)} shared recordings")
        if overlap:
            print("  Examples:", sorted(overlap)[:10])

print("\n" + "=" * 80)
print("3. C2 SPLIT LABELS AND DATASET SUPPORT")

for name in ["C2_train", "C2_validation", "C2_ICBHI_test", "C2_Fraiwan_test"]:
    if name not in manifests:
        continue
    df = manifests[name]
    print(f"\n{name}")
    print("Recorded split values:")
    print(df["split"].value_counts(dropna=False).to_string() if "split" in df else "No split column")
    if "source_dataset" in df:
        print("Recordings by source:")
        print(df["source_dataset"].value_counts(dropna=False).to_string())
    if "diagnosis" in df:
        print("Unique patients by diagnosis:")
        print(
            df.groupby("diagnosis")["patient_id"].nunique()
            .sort_values(ascending=False).to_string()
            if "patient_id" in df else "No patient_id column"
        )

print("\nAudit complete. No files were modified.")


In [ ]:

from pathlib import Path
import pandas as pd

BASE = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
C2 = BASE / "experiment_c2_icbhi_fraiwan_high_capacity"

print("C2 directory:", C2)
print("Directory exists:", C2.exists())

print("\n" + "=" * 85)
print("1. FILE INVENTORY")

for path in sorted(C2.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(C2)}  |  {path.stat().st_size / 1024:.1f} KB")

print("\n" + "=" * 85)
print("2. CSV SUMMARY FILES")

csv_files = sorted(C2.rglob("*.csv"))
for path in csv_files:
    name = path.name.lower()
    # Prioritize metrics, predictions, histories, features, and metadata.
    if any(term in name for term in [
        "summary", "result", "metric", "history", "prediction",
        "feature", "evaluation", "report", "config", "epoch"
    ]):
        try:
            df = pd.read_csv(path)
            print(f"\n--- {path.relative_to(C2)} | shape={df.shape} ---")
            print("Columns:", df.columns.tolist())
            print(df.head(8).to_string(index=False))
        except Exception as exc:
            print(f"\nCould not read {path.name}: {exc}")

print("\n" + "=" * 85)
print("3. MODEL / CHECKPOINT FILES")

model_extensions = {
    ".pt", ".pth", ".keras", ".h5", ".joblib", ".pkl",
    ".onnx", ".safetensors", ".json", ".ubj", ".model"
}
found_models = []
for path in sorted(C2.rglob("*")):
    if path.is_file() and path.suffix.lower() in model_extensions:
        found_models.append(path)
        print(f"{path.relative_to(C2)}  |  {path.stat().st_size / (1024 * 1024):.2f} MB")

if not found_models:
    print("No model/checkpoint files found inside C2.")

print("\nInventory complete. No files were changed.")


In [ ]:

import json
from pathlib import Path
import numpy as np

C2 = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity"

print("=" * 80)
print("1. CNN TRAINING HISTORY")
history_path = C2 / "c2_compact_multifeature_training_history.json"

if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)

    if isinstance(history, dict):
        print("Top-level keys:", list(history.keys()))
        for key, value in history.items():
            if isinstance(value, list):
                print(f"\n{key}: {len(value)} entries")
                print("First 3:", value[:3])
                print("Last 5:", value[-5:])
            elif isinstance(value, dict):
                print(f"\n{key}:")
                print(json.dumps(value, indent=2)[:4000])
            else:
                print(f"{key}: {value}")
    else:
        print(str(history)[:6000])
else:
    print("Training history not found.")

print("\n" + "=" * 80)
print("2. FEATURE EXTRACTION AND NORMALIZATION CONFIGS")

config_paths = [
    C2 / "features_logmel_v1" / "feature_extraction_config.json",
    C2 / "features_logmel_v1_normalized" / "normalization_config.json",
    C2 / "features_multifeature_v1_normalized" / "normalization_config.json",
    C2 / "features_multifeature_v2_normalized" / "normalization_config.json",
]

for path in config_paths:
    print(f"\n--- {path.relative_to(C2)} ---")
    if path.exists():
        try:
            config = json.loads(path.read_text())
            print(json.dumps(config, indent=2)[:5000])
        except Exception as exc:
            print("Could not parse:", exc)
    else:
        print("Not found")

print("\n" + "=" * 80)
print("3. FEATURE CACHE KEYS AND SHAPES")

cache_paths = [
    C2 / "features_multifeature_v2" / "train_features.npz",
    C2 / "features_multifeature_v2" / "validation_features.npz",
    C2 / "features_multifeature_v2" / "icbhi_test_features.npz",
    C2 / "features_multifeature_v2" / "fraiwan_test_features.npz",
    C2 / "features_multifeature_v2_normalized" / "train_features_normalized.npz",
    C2 / "features_multifeature_v2_normalized" / "validation_features_normalized.npz",
]

for path in cache_paths:
    print(f"\n--- {path.relative_to(C2)} ---")
    if path.exists():
        try:
            with np.load(path, allow_pickle=False) as data:
                for key in data.files:
                    arr = data[key]
                    print(
                        f"{key}: shape={arr.shape}, dtype={arr.dtype}, "
                        f"finite={bool(np.isfinite(arr).all()) if np.issubdtype(arr.dtype, np.number) else 'n/a'}"
                    )
        except Exception as exc:
            print("Could not inspect:", exc)
    else:
        print("Not found")

print("\n" + "=" * 80)
print("4. CHECKPOINT METADATA")

checkpoint = C2 / "c2_compact_multifeature_best.pt"
print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    try:
        import torch
        try:
            obj = torch.load(checkpoint, map_location="cpu", weights_only=True)
        except TypeError:
            print("This PyTorch version does not support weights_only=True.")
            obj = None

        if isinstance(obj, dict):
            print("Checkpoint keys:", list(obj.keys()))
            for key, value in obj.items():
                if isinstance(value, torch.Tensor):
                    print(f"{key}: tensor shape={tuple(value.shape)}, dtype={value.dtype}")
                elif isinstance(value, dict):
                    print(f"{key}: nested dict with {len(value)} entries")
                elif isinstance(value, (str, int, float, bool, list, tuple)):
                    print(f"{key}: {str(value)[:500]}")
                else:
                    print(f"{key}: type={type(value).__name__}")
        elif obj is not None:
            print("Loaded object type:", type(obj).__name__)
        else:
            print("Checkpoint not loaded; no changes made.")
    except Exception as exc:
        print("Checkpoint inspection issue:", repr(exc))

print("\nInspection complete. No files were modified.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

C2 = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity"
FEATURES = C2 / "features_multifeature_v2"
NORMALIZED = C2 / "features_multifeature_v2_normalized"

expected = {
    "train": (FEATURES / "train_features.npz", FEATURES / "train_metadata.csv",
              NORMALIZED / "train_features_normalized.npz"),
    "validation": (FEATURES / "validation_features.npz", FEATURES / "validation_metadata.csv",
                   NORMALIZED / "validation_features_normalized.npz"),
    "icbhi_test": (FEATURES / "icbhi_test_features.npz", FEATURES / "icbhi_test_metadata.csv",
                   NORMALIZED / "icbhi_test_features_normalized.npz"),
    "fraiwan_test": (FEATURES / "fraiwan_test_features.npz", FEATURES / "fraiwan_test_metadata.csv",
                     NORMALIZED / "fraiwan_test_features_normalized.npz"),
}

all_labels = set()
metadata_frames = {}

for split, (feature_path, metadata_path, normalized_path) in expected.items():
    print("\n" + "=" * 80)
    print(split.upper())

    print("Feature file exists:", feature_path.exists())
    print("Metadata file exists:", metadata_path.exists())
    print("Normalized feature file exists:", normalized_path.exists())

    if not feature_path.exists() or not metadata_path.exists():
        print("Skipping: required feature or metadata file missing.")
        continue

    metadata = pd.read_csv(metadata_path)
    metadata_frames[split] = metadata

    print("Metadata shape:", metadata.shape)
    print("Metadata columns:", metadata.columns.tolist())

    if "diagnosis" in metadata.columns:
        print("\nDiagnosis counts:")
        print(metadata["diagnosis"].value_counts(dropna=False).to_string())
        all_labels.update(metadata["diagnosis"].dropna().astype(str).unique())

    with np.load(feature_path, allow_pickle=False) as data:
        print("\nRaw feature row counts:")
        for key in data.files:
            print(f"  {key}: {data[key].shape[0]} rows")

        if "logmel" in data.files:
            n_feature_rows = data["logmel"].shape[0]
            print("Log-Mel rows match metadata:", n_feature_rows == len(metadata))

    if normalized_path.exists():
        with np.load(normalized_path, allow_pickle=False) as data:
            print("\nNormalized feature row counts:")
            for key in data.files:
                print(f"  {key}: {data[key].shape[0]} rows")
                if key in ("logmel", "mfcc", "chroma"):
                    print(f"    finite values: {bool(np.isfinite(data[key]).all())}")

    identity_cols = [
        c for c in ["source_dataset", "source_recording_id", "patient_id",
                    "diagnosis", "recording_path"]
        if c in metadata.columns
    ]
    print("\nFirst 5 metadata identity rows:")
    print(metadata[identity_cols].head().to_string(index=False))

print("\n" + "=" * 80)
print("LABEL VOCABULARY")

print("All labels seen across available feature metadata:")
print(sorted(all_labels))

expected_eight = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]
print("Missing from expected eight-class vocabulary:",
      sorted(set(expected_eight) - all_labels))
print("Unexpected labels:",
      sorted(all_labels - set(expected_eight)))

print("\nAudit complete. No files were modified.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

C2 = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity"
FEATURES = C2 / "features_multifeature_v2_normalized"

split_files = {
    "train": (FEATURES / "train_features_normalized.npz",
              C2 / "features_multifeature_v2" / "train_metadata.csv"),
    "validation": (FEATURES / "validation_features_normalized.npz",
                   C2 / "features_multifeature_v2" / "validation_metadata.csv"),
    "icbhi_test": (FEATURES / "icbhi_test_features_normalized.npz",
                   C2 / "features_multifeature_v2" / "icbhi_test_metadata.csv"),
    "fraiwan_test": (FEATURES / "fraiwan_test_features_normalized.npz",
                     C2 / "features_multifeature_v2" / "fraiwan_test_metadata.csv"),
}

def summarize_feature_map(arr, name):
    # arr shape: (recordings, channels, time)
    return np.concatenate(
        [arr.mean(axis=2), arr.std(axis=2)],
        axis=1
    )

recording_data = {}

for split, (feature_path, metadata_path) in split_files.items():
    print("\n" + "=" * 75)
    print(split.upper())

    if not feature_path.exists() or not metadata_path.exists():
        raise FileNotFoundError(
            f"Missing input for {split}: {feature_path} or {metadata_path}"
        )

    metadata = pd.read_csv(metadata_path).reset_index(drop=True)

    with np.load(feature_path, allow_pickle=False) as data:
        required = {"logmel", "mfcc", "chroma"}
        missing = required - set(data.files)
        if missing:
            raise ValueError(f"{split}: missing feature arrays {missing}")

        n_rows = len(metadata)
        for name in sorted(required):
            if data[name].shape[0] != n_rows:
                raise ValueError(
                    f"{split}: {name} rows {data[name].shape[0]} "
                    f"!= metadata rows {n_rows}"
                )
            if not np.isfinite(data[name]).all():
                raise ValueError(f"{split}: non-finite values in {name}")

        X = np.concatenate([
            summarize_feature_map(data["logmel"], "logmel"),
            summarize_feature_map(data["mfcc"], "mfcc"),
            summarize_feature_map(data["chroma"], "chroma"),
        ], axis=1).astype(np.float32)

    if X.shape[1] != 178:
        raise ValueError(f"Expected 178 features, got {X.shape[1]}")

    if not {"diagnosis", "patient_id", "source_dataset", "source_recording_id"}.issubset(metadata.columns):
        raise ValueError(f"{split}: required identity/label columns missing")

    recording_data[split] = {
        "X": X,
        "metadata": metadata,
        "y": metadata["diagnosis"].astype(str).to_numpy(),
        "groups": (
            metadata["source_dataset"].astype(str)
            + "::"
            + metadata["patient_id"].astype(str)
        ).to_numpy(),
    }

    print("Feature matrix:", X.shape)
    print("Metadata rows:", len(metadata))
    print("All feature values finite:", bool(np.isfinite(X).all()))
    print("Unique patients:", metadata[["source_dataset", "patient_id"]].drop_duplicates().shape[0])
    print("Class counts:")
    print(metadata["diagnosis"].value_counts().to_string())

print("\n" + "=" * 75)
print("SPLIT OVERLAP CHECK")

for left, right in [
    ("train", "validation"),
    ("train", "icbhi_test"),
    ("train", "fraiwan_test"),
    ("validation", "icbhi_test"),
    ("validation", "fraiwan_test"),
]:
    a = set(recording_data[left]["groups"])
    b = set(recording_data[right]["groups"])
    print(f"{left} vs {right}: {len(a & b)} shared patient IDs")

print("\nFeature preparation audit complete; nothing was saved.")


In [ ]:

import numpy as np
import pandas as pd
import time
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=UserWarning)

# Requires recording_data from the previous feature-preparation cell.
required_splits = {"train", "validation", "icbhi_test", "fraiwan_test"}
if "recording_data" not in globals() or not required_splits.issubset(recording_data):
    raise RuntimeError("Run the feature-preparation audit cell first.")

EIGHT_CLASSES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]
FIVE_CLASSES = [
    "COPD", "Healthy", "Pneumonia", "Bronchiectasis", "URTI"
]

X_train_all = recording_data["train"]["X"]
y_train_all = recording_data["train"]["y"]
X_val_all = recording_data["validation"]["X"]
y_val_all = recording_data["validation"]["y"]

# Optional XGBoost support.
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed; the other models will still be compared.")

def supported_macro_f1(y_true, y_pred, labels):
    present = [label for label in labels if np.any(y_true == label)]
    return f1_score(
        y_true, y_pred, labels=present,
        average="macro", zero_division=0
    )

def build_candidates(seed=42):
    candidates = []

    for C in [0.1, 1.0, 10.0]:
        candidates.append((
            f"LogisticRegression_C={C}",
            make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                LogisticRegression(
                    C=C, class_weight="balanced",
                    max_iter=3000, random_state=seed
                )
            )
        ))

    for C in [0.1, 1.0, 10.0]:
        candidates.append((
            f"SVM_RBF_C={C}",
            make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                SVC(
                    C=C, kernel="rbf", class_weight="balanced",
                    probability=True, random_state=seed
                )
            )
        ))

    for depth in [None, 12]:
        candidates.append((
            f"RandomForest_depth={depth}",
            make_pipeline(
                SimpleImputer(strategy="median"),
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=depth,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=seed,
                    n_jobs=-1
                )
            )
        ))

    if XGBOOST_AVAILABLE:
        for depth in [2, 3]:
            candidates.append((
                f"XGBoost_depth={depth}",
                make_pipeline(
                    SimpleImputer(strategy="median"),
                    XGBClassifier(
                        n_estimators=250,
                        learning_rate=0.04,
                        max_depth=depth,
                        min_child_weight=3,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        reg_lambda=5.0,
                        objective="multi:softprob",
                        eval_metric="mlogloss",
                        tree_method="hist",
                        random_state=seed,
                        n_jobs=-1
                    )
                )
            ))

    return candidates

selection_rows = []
selected_models = {}
validation_predictions = {}

for task_name, target_classes in [
    ("8_class", EIGHT_CLASSES),
    ("5_class", FIVE_CLASSES),
]:
    print("\n" + "=" * 90)
    print(f"TASK: {task_name} | classes={target_classes}")

    train_mask = np.isin(y_train_all, target_classes)
    val_mask = np.isin(y_val_all, target_classes)

    Xtr = X_train_all[train_mask]
    ytr = y_train_all[train_mask]
    Xv = X_val_all[val_mask]
    yv = y_val_all[val_mask]

    missing_train = sorted(set(target_classes) - set(ytr))
    if missing_train:
        raise ValueError(f"{task_name}: classes missing from training: {missing_train}")

    print("Training recordings:", len(ytr))
    print("Validation recordings:", len(yv))
    print("Training support:")
    print(pd.Series(ytr).value_counts().reindex(target_classes, fill_value=0).to_string())
    print("Validation support:")
    print(pd.Series(yv).value_counts().reindex(target_classes, fill_value=0).to_string())

    encoder = LabelEncoder().fit(target_classes)
    ytr_encoded = encoder.transform(ytr)

    for model_name, model in build_candidates():
        start = time.time()

        # XGBoost needs integer targets. Other estimators can use string labels.
        if model_name.startswith("XGBoost"):
            model.fit(Xtr, ytr_encoded)
            pred_encoded = model.predict(Xv).astype(int)
            pred = encoder.inverse_transform(pred_encoded)
        else:
            model.fit(Xtr, ytr)
            pred = model.predict(Xv)

        elapsed = time.time() - start

        row = {
            "task": task_name,
            "model": model_name,
            "validation_accuracy": accuracy_score(yv, pred),
            "validation_balanced_accuracy": balanced_accuracy_score(yv, pred),
            "validation_macro_f1_all_target_classes": f1_score(
                yv, pred, labels=target_classes,
                average="macro", zero_division=0
            ),
            "validation_macro_f1_supported_classes": supported_macro_f1(
                yv, pred, target_classes
            ),
            "fit_predict_seconds": round(elapsed, 2),
        }
        selection_rows.append(row)

        # Keep models and predictions in memory for the selected configuration.
        validation_predictions[(task_name, model_name)] = pred
        selected_models[(task_name, model_name)] = model

    task_results = pd.DataFrame(
        [r for r in selection_rows if r["task"] == task_name]
    ).sort_values(
        "validation_macro_f1_all_target_classes", ascending=False
    )

    print("\nValidation ranking:")
    print(task_results.to_string(index=False))

    best = task_results.iloc[0]
    print(
        f"\nBest by validation all-class macro-F1: {best['model']} "
        f"(F1={best['validation_macro_f1_all_target_classes']:.4f})"
    )

selection_table = pd.DataFrame(selection_rows).sort_values(
    ["task", "validation_macro_f1_all_target_classes"],
    ascending=[True, False]
)

print("\n" + "=" * 90)
print("FINAL VALIDATION-ONLY MODEL RANKING")
display(selection_table.reset_index(drop=True))

print(
    "\nNo test predictions were made. No files were saved. "
    "These are validation-selected candidates, not final test results."
)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

C2 = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity"
FEATURES = C2 / "features_multifeature_v2_normalized"

split_files = {
    "train": (
        FEATURES / "train_features_normalized.npz",
        C2 / "features_multifeature_v2" / "train_metadata.csv",
    ),
    "validation": (
        FEATURES / "validation_features_normalized.npz",
        C2 / "features_multifeature_v2" / "validation_metadata.csv",
    ),
    "icbhi_test": (
        FEATURES / "icbhi_test_features_normalized.npz",
        C2 / "features_multifeature_v2" / "icbhi_test_metadata.csv",
    ),
    "fraiwan_test": (
        FEATURES / "fraiwan_test_features_normalized.npz",
        C2 / "features_multifeature_v2" / "fraiwan_test_metadata.csv",
    ),
}

def summarize_feature_map(arr):
    return np.concatenate(
        [arr.mean(axis=2), arr.std(axis=2)],
        axis=1,
    )

recording_data = {}

for split, (feature_path, metadata_path) in split_files.items():
    if not feature_path.exists():
        raise FileNotFoundError(f"Feature cache missing: {feature_path}")
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata missing: {metadata_path}")

    metadata = pd.read_csv(metadata_path).reset_index(drop=True)

    with np.load(feature_path, allow_pickle=False) as data:
        X = np.concatenate(
            [
                summarize_feature_map(data["logmel"]),
                summarize_feature_map(data["mfcc"]),
                summarize_feature_map(data["chroma"]),
            ],
            axis=1,
        ).astype(np.float32)

    if len(metadata) != len(X):
        raise ValueError(
            f"{split}: {len(metadata)} metadata rows but {len(X)} feature rows"
        )

    recording_data[split] = {
        "X": X,
        "metadata": metadata,
        "y": metadata["diagnosis"].astype(str).to_numpy(),
        "groups": (
            metadata["source_dataset"].astype(str)
            + "::"
            + metadata["patient_id"].astype(str)
        ).to_numpy(),
    }

    print(
        f"{split}: X={X.shape}, "
        f"recordings={len(metadata)}, "
        f"patients={metadata[['source_dataset', 'patient_id']].drop_duplicates().shape[0]}"
    )

required_splits = {"train", "validation", "icbhi_test", "fraiwan_test"}
assert required_splits.issubset(recording_data)
print("\nReload successful. recording_data is ready.")


In [ ]:

from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

C2 = (
    PROJECT_ROOT
    / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c2_icbhi_fraiwan_high_capacity"
)

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.is_dir())
print("C2 directory exists:", C2.is_dir())

assert PROJECT_ROOT.is_dir(), "Project path not found. Check the folder location."
assert C2.is_dir(), "C2 experiment directory not found."

print("Setup successful.")


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "selected_models" not in globals() or not selected_models:
    raise RuntimeError("Selected fitted models not found. Rerun the model-selection cell first.")

EIGHT_CLASSES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]
FIVE_CLASSES = [
    "COPD", "Healthy", "Pneumonia", "Bronchiectasis", "URTI"
]

test_splits = ["icbhi_test", "fraiwan_test"]
test_summary_rows = []
test_class_reports = {}
test_confusion_matrices = {}

for task_name, target_classes in [
    ("8_class", EIGHT_CLASSES),
    ("5_class", FIVE_CLASSES),
]:
    print("\n" + "#" * 100)
    print("TASK:", task_name)

    for test_split in test_splits:
        X_test_all = recording_data[test_split]["X"]
        y_test_all = recording_data[test_split]["y"]

        mask = np.isin(y_test_all, target_classes)
        X_test = X_test_all[mask]
        y_test = y_test_all[mask]

        print("\n" + "=" * 90)
        print(f"{test_split.upper()} | {len(y_test)} recordings in this task")
        print("Test support:")
        print(pd.Series(y_test).value_counts().reindex(target_classes, fill_value=0).to_string())

        if len(y_test) == 0:
            print("No recordings belong to this task's class subset; skipped.")
            continue

        for model_name, model in [
            (name, selected_models[(task_name, name)])
            for t, name in selected_models
            if t == task_name
        ]:
            # XGBoost was trained using LabelEncoder targets.
            if model_name.startswith("XGBoost"):
                encoder = __import__("sklearn.preprocessing", fromlist=["LabelEncoder"]).LabelEncoder()
                encoder.fit(target_classes)
                pred_encoded = model.predict(X_test).astype(int)
                y_pred = encoder.inverse_transform(pred_encoded)
            else:
                y_pred = model.predict(X_test)

            supported_classes = [
                c for c in target_classes if np.any(y_test == c)
            ]

            row = {
                "task": task_name,
                "test_dataset": test_split,
                "model": model_name,
                "n_test_recordings": len(y_test),
                "n_supported_classes": len(supported_classes),
                "accuracy": accuracy_score(y_test, y_pred),
                "balanced_accuracy_supported_classes": balanced_accuracy_score(
                    y_test, y_pred
                ),
                "macro_f1_all_target_classes": f1_score(
                    y_test, y_pred,
                    labels=target_classes,
                    average="macro",
                    zero_division=0,
                ),
                "macro_f1_supported_classes": f1_score(
                    y_test, y_pred,
                    labels=supported_classes,
                    average="macro",
                    zero_division=0,
                ),
                "weighted_f1": f1_score(
                    y_test, y_pred,
                    labels=target_classes,
                    average="weighted",
                    zero_division=0,
                ),
            }
            test_summary_rows.append(row)

            key = (task_name, test_split, model_name)
            test_class_reports[key] = pd.DataFrame(
                classification_report(
                    y_test,
                    y_pred,
                    labels=target_classes,
                    output_dict=True,
                    zero_division=0,
                )
            ).T
            test_confusion_matrices[key] = pd.DataFrame(
                confusion_matrix(
                    y_test, y_pred, labels=target_classes
                ),
                index=[f"true_{c}" for c in target_classes],
                columns=[f"pred_{c}" for c in target_classes],
            )

test_summary = pd.DataFrame(test_summary_rows)

print("\n" + "#" * 100)
print("TEST SUMMARY — DESCRIPTIVE EVALUATION ONLY")
if not test_summary.empty:
    display(
        test_summary.sort_values(
            ["task", "test_dataset", "macro_f1_all_target_classes"],
            ascending=[True, True, False],
        ).reset_index(drop=True)
    )

# Show detailed class reports for each candidate.
for key, report in test_class_reports.items():
    print("\n" + "=" * 90)
    print("Per-class report:", key)
    display(report)

print(
    "\nEvaluation complete. No test-driven tuning, refitting, or file saving was performed."
)


In [ ]:
# Check that the classical ML evaluation outputs are still in memory.
required_objects = [
    "test_summary",
    "test_class_reports",
    "test_confusion_matrices",
    "recording_data",
    "selected_models",
]

for name in required_objects:
    obj = globals().get(name)
    print(f"{name}: {'available' if obj is not None else 'MISSING'}")

if "test_summary" in globals():
    display(test_summary.head())

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path

C2 = PROJECT_ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity"

checkpoint_path = C2 / "c2_compact_multifeature_best.pt"
feature_dir = C2 / "features_multifeature_v2_normalized"

print("PyTorch:", torch.__version__)
print("Checkpoint exists:", checkpoint_path.exists())
print("Feature directory exists:", feature_dir.is_dir())

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nArchitecture:", checkpoint["architecture"])
print("Classes:", checkpoint["class_names"])
print("Best epoch:", checkpoint["best_epoch"])
print("Validation macro-F1:", checkpoint["best_validation_macro_f1_supported_classes"])
print("Trainable parameters:", checkpoint["trainable_parameters"])

for split in ["icbhi_test", "fraiwan_test"]:
    path = feature_dir / f"{split}_features_normalized.npz"
    print(f"\n{split}: exists={path.exists()}")
    if path.exists():
        with np.load(path) as data:
            print("Feature keys:", data.files)
            for key in data.files:
                print(f"  {key}: {data[key].shape}, finite={np.isfinite(data[key]).all()}")

In [ ]:
import ast

search_roots = [
    PROJECT_ROOT / "Notebooks 2",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "scripts",
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.suffix == ".py":
            try:
                source = path.read_text(encoding="utf-8")
                if "CompactMultiFeatureCNN" in source:
                    matches.append((path, source))
            except Exception:
                pass

        elif path.suffix == ".ipynb":
            try:
                import nbformat
                notebook = nbformat.read(path, as_version=4)

                for cell_index, cell in enumerate(notebook.cells):
                    if cell.cell_type == "code" and "CompactMultiFeatureCNN" in cell.source:
                        matches.append(
                            (f"{path} [cell {cell_index}]", cell.source)
                        )
            except Exception:
                pass

print("Matching files/cells:", len(matches))

for path, source in matches:
    print("\n" + "=" * 90)
    print("SOURCE:", path)

    lines = source.splitlines()
    for i, line in enumerate(lines):
        if "class CompactMultiFeatureCNN" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 100)
            print("\n".join(f"{j+1}: {lines[j]}" for j in range(start, end)))
            break

In [ ]:
import nbformat

source_notebook = (
    PROJECT_ROOT
    / "Notebooks 2"
    / "08_Combined_ICBHI_Fraiwan_HighCapacity_CNN.ipynb"
)

nb = nbformat.read(source_notebook, as_version=4)

for cell_index, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    if "class CompactFeatureBranch" in cell.source:
        print(f"Found definition in cell {cell_index}\n")
        print(cell.source)
        break
else:
    print("CompactFeatureBranch definition not found in this notebook.")

In [ ]:
import torch
import torch.nn as nn

# Use the same architecture as the original training notebook.
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")


class CompactFeatureBranch(nn.Module):
    def __init__(self, out_channels=24):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 12, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(12),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(12, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, groups=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.features(x.unsqueeze(1))


class CompactMultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8, dropout=0.30):
        super().__init__()

        self.logmel_branch = CompactFeatureBranch(out_channels=24)
        self.mfcc_branch = CompactFeatureBranch(out_channels=16)
        self.chroma_branch = CompactFeatureBranch(out_channels=16)

        self.classifier = nn.Sequential(
            nn.Linear(24 + 16 + 16, 48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        mel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        fused = torch.cat(
            [mel_features, mfcc_features, chroma_features], dim=1
        )
        return self.classifier(fused)


# Load the existing checkpoint.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

CNN_CLASS_NAMES = checkpoint["class_names"]

cnn_model = CompactMultiFeatureCNN(
    num_classes=len(CNN_CLASS_NAMES)
).to(DEVICE)

# Strict loading ensures every expected parameter matches.
cnn_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
cnn_model.eval()

trainable_params = sum(
    p.numel() for p in cnn_model.parameters() if p.requires_grad
)

print("Device:", DEVICE)
print("Architecture:", checkpoint["architecture"])
print("Classes:", CNN_CLASS_NAMES)
print("Checkpoint epoch:", checkpoint["best_epoch"])
print("Trainable parameters:", f"{trainable_params:,}")
print("Checkpoint loaded strictly: YES")

assert trainable_params == checkpoint["trainable_parameters"] == 19588

print("\nCNN architecture and checkpoint verification passed.")

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

CLASS_NAMES_CNN = list(CNN_CLASS_NAMES)

def evaluate_cnn_test_split(split_name):
    feature_path = feature_dir / f"{split_name}_features_normalized.npz"
    metadata_path = C2 / "features_multifeature_v2" / f"{split_name}_metadata.csv"

    if not feature_path.exists():
        raise FileNotFoundError(feature_path)
    if not metadata_path.exists():
        raise FileNotFoundError(metadata_path)

    with np.load(feature_path) as data:
        X_logmel = data["logmel"].copy().astype(np.float32)
        X_mfcc = data["mfcc"].copy().astype(np.float32)
        X_chroma = data["chroma"].copy().astype(np.float32)

    metadata = pd.read_csv(metadata_path)

    # Confirm metadata rows correspond to cached feature rows.
    assert len(metadata) == len(X_logmel), (
        f"{split_name}: metadata rows ({len(metadata)}) "
        f"do not match features ({len(X_logmel)})"
    )

    # Inspect the label column rather than assuming its name.
    possible_label_cols = [
        c for c in metadata.columns
        if c.lower() in {"diagnosis", "label", "class", "class_name"}
    ]
    if not possible_label_cols:
        raise ValueError(
            f"Could not identify diagnosis column in {split_name} metadata. "
            f"Columns: {metadata.columns.tolist()}"
        )

    label_col = possible_label_cols[0]
    y_true_names = metadata[label_col].astype(str).to_numpy()

    unknown = sorted(set(y_true_names) - set(CLASS_NAMES_CNN))
    if unknown:
        raise ValueError(f"Unknown labels in {split_name}: {unknown}")

    class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES_CNN)}
    y_true = np.array([class_to_idx[name] for name in y_true_names])

    predictions = []
    batch_size = 32

    with torch.inference_mode():
        for start in range(0, len(X_logmel), batch_size):
            end = start + batch_size

            logmel_batch = torch.from_numpy(X_logmel[start:end]).to(DEVICE)
            mfcc_batch = torch.from_numpy(X_mfcc[start:end]).to(DEVICE)
            chroma_batch = torch.from_numpy(X_chroma[start:end]).to(DEVICE)

            logits = cnn_model(
                logmel_batch, mfcc_batch, chroma_batch
            )
            predictions.extend(logits.argmax(dim=1).cpu().numpy())

    y_pred = np.asarray(predictions)

    metrics = {
        "split": split_name,
        "n_recordings": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_all_classes_present": balanced_accuracy_score(
            y_true, y_pred
        ),
        "macro_f1_all_8_classes": f1_score(
            y_true, y_pred,
            labels=np.arange(len(CLASS_NAMES_CNN)),
            average="macro",
            zero_division=0,
        ),
        "macro_f1_supported_classes": f1_score(
            y_true, y_pred,
            labels=np.unique(y_true),
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES_CNN)),
        target_names=CLASS_NAMES_CNN,
        output_dict=True,
        zero_division=0,
    )

    per_class = pd.DataFrame(report).T
    cm = confusion_matrix(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES_CNN))
    )

    print(f"\n{'=' * 70}")
    print(f"CNN TEST RESULTS — {split_name}")
    print(f"{'=' * 70}")
    print(pd.Series(metrics).to_string())
    print("\nPer-class classification report:")
    display(per_class.round(4))
    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(
        cm,
        index=CLASS_NAMES_CNN,
        columns=CLASS_NAMES_CNN
    ))

    return metrics, per_class, cm, y_true, y_pred


cnn_icbhi_results = evaluate_cnn_test_split("icbhi_test")
cnn_fraiwan_results = evaluate_cnn_test_split("fraiwan_test")

In [ ]:
from pathlib import Path

search_roots = [
    PROJECT_ROOT / "Notebooks 2",
    PROJECT_ROOT / "data" / "processed" / "deep_learning",
    PROJECT_ROOT / "models",
]

keywords = (
    "cnn", "lstm", "logmel", "checkpoint",
    "best.pt", "best.pth", "history.json"
)

print("=" * 85)
print("CNN MODEL AND CHECKPOINT AUDIT")
print("=" * 85)

for root in search_roots:
    if not root.exists():
        print(f"\nDirectory not found: {root}")
        continue

    print(f"\n--- {root.relative_to(PROJECT_ROOT)} ---")

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        if any(term in path.name.lower() for term in keywords):
            try:
                size_mb = path.stat().st_size / (1024 ** 2)
                print(
                    f"{path.relative_to(PROJECT_ROOT)}"
                    f"  |  {size_mb:.3f} MB"
                )
            except OSError:
                print(path)

print("\nAudit complete. No files were modified.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = PROJECT_ROOT / "data" / "processed" / "deep_learning"

files_to_inspect = [
    BASE / "cnn_baseline" / "cnn_architecture.json",
    BASE / "cnn_baseline" / "cnn_training_config.json",
    BASE / "cnn_baseline" / "cnn_baseline_metrics.csv",
    BASE / "cnn_baseline" / "cnn_baseline_experiment_summary.csv",
    BASE / "cnn_lstm" / "cnn_lstm_training_config.json",
    BASE / "cnn_lstm" / "cnn_lstm_experiment_summary.json",
    BASE / "cnn_lstm" / "cnn_lstm_metrics.csv",
    BASE / "cnn_lstm" / "cnn_model_comparison.csv",
]

for path in files_to_inspect:
    print("\n" + "=" * 85)
    print(path.relative_to(PROJECT_ROOT))

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        if path.suffix == ".json":
            print(json.dumps(json.loads(path.read_text()), indent=2))
        elif path.suffix == ".csv":
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            display(df.head(20))
        else:
            print("Unsupported file type:", path.suffix)
    except Exception as exc:
        print("Could not read:", repr(exc))

In [ ]:
import nbformat
import re

notebook_paths = [
    PROJECT_ROOT / "Notebooks 2" / "05_Lightweight_MultiFeature_CNN.ipynb",
    PROJECT_ROOT / "Notebooks 2" / "08_Combined_ICBHI_Fraiwan_HighCapacity_CNN.ipynb",
]

for path in notebook_paths:
    print("\n" + "=" * 90)
    print("NOTEBOOK:", path.name)

    if not path.exists():
        print("Not found")
        continue

    nb = nbformat.read(path, as_version=4)

    for cell_idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue

        source = cell.source
        relevant = (
            re.search(r"class\s+\w*(CNN|LSTM)\w*\s*\(", source, re.I)
            or "trainable_params" in source
            or "total_params" in source
            or "best_epoch" in source
        )

        if relevant:
            print(f"\n--- Cell {cell_idx} ---")
            lines = source.splitlines()

            for line_no, line in enumerate(lines):
                if (
                    re.search(r"class\s+\w*(CNN|LSTM)\w*\s*\(", line, re.I)
                    or "trainable_params" in line
                    or "total_params" in line
                    or "best_epoch" in line
                    or "Conv2d(" in line
                    or "LSTM(" in line
                    or "Linear(" in line
                ):
                    start = max(0, line_no - 1)
                    end = min(len(lines), line_no + 3)
                    print("\n".join(lines[start:end]))
                    print()

In [ ]:
from pathlib import Path
import torch
import json

BASE = PROJECT_ROOT / "data" / "processed" / "deep_learning"

checkpoint_paths = [
    BASE / "cnn_baseline" / "best_lightweight_mel_cnn.pt",
    BASE / "lightweight_multifeature_cnn" / "best_lightweight_multifeature_cnn.pt",
    BASE / "multifeature_cnn" / "best_multifeature_cnn.pt",
    BASE / "multifeature_cnn_12b" / "best_regularized_multifeature_cnn.pt",
    BASE / "cnn_lstm" / "best_cnn_lstm.pt",
    BASE / "combined_icbhi_fraiwan_hf_lung_v1"
         / "experiment_c2_icbhi_fraiwan_high_capacity"
         / "c2_compact_multifeature_best.pt",
]

for path in checkpoint_paths:
    print("\n" + "=" * 85)
    print("CHECKPOINT:", path.relative_to(PROJECT_ROOT))

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)

        if isinstance(obj, dict):
            print("Top-level keys:", list(obj.keys()))

            for key in [
                "architecture", "model_name", "class_names", "classes",
                "trainable_parameters", "best_epoch",
                "best_validation_macro_f1_supported_classes",
                "best_validation_macro_f1_supported",
                "best_validation_loss", "num_classes",
                "sample_rate", "target_sample_rate",
            ]:
                if key in obj:
                    print(f"{key}: {obj[key]}")

            state = obj.get("model_state_dict", obj.get("state_dict"))
            if isinstance(state, dict):
                print("State-dict tensors:", len(state))
                print("First parameter keys:", list(state.keys())[:8])
            elif all(torch.is_tensor(v) for v in obj.values()):
                print("Raw state-dict tensors:", len(obj))
                print("First parameter keys:", list(obj.keys())[:8])
        else:
            print("Object type:", type(obj).__name__)

    except Exception as exc:
        print("Could not inspect checkpoint:", repr(exc))

print("\nCheckpoint inspection complete. No files were modified.")

In [ ]:
from pathlib import Path
import json
import torch

BASE = PROJECT_ROOT / "data" / "processed" / "deep_learning"

checkpoint_paths = {
    "Log-Mel CNN": BASE / "cnn_baseline" / "best_lightweight_mel_cnn.pt",
    "Lightweight multi-feature CNN": (
        BASE / "lightweight_multifeature_cnn"
        / "best_lightweight_multifeature_cnn.pt"
    ),
    "Multi-feature CNN": BASE / "multifeature_cnn" / "best_multifeature_cnn.pt",
    "Regularized multi-feature CNN": (
        BASE / "multifeature_cnn_12b"
        / "best_regularized_multifeature_cnn.pt"
    ),
    "CNN-LSTM": BASE / "cnn_lstm" / "best_cnn_lstm.pt",
    "Compact multi-feature CNN": (
        BASE / "combined_icbhi_fraiwan_hf_lung_v1"
        / "experiment_c2_icbhi_fraiwan_high_capacity"
        / "c2_compact_multifeature_best.pt"
    ),
}

for name, path in checkpoint_paths.items():
    print("\n" + "=" * 75)
    print(name)

    if not path.exists():
        print("Checkpoint missing")
        continue

    obj = torch.load(path, map_location="cpu", weights_only=False)

    if "model_state_dict" in obj:
        state = obj["model_state_dict"]
    elif "state_dict" in obj:
        state = obj["state_dict"]
    else:
        state = obj

    if isinstance(state, dict) and all(torch.is_tensor(v) for v in state.values()):
        parameter_count = sum(
            tensor.numel()
            for key, tensor in state.items()
            if not key.endswith("num_batches_tracked")
            and not key.endswith("running_mean")
            and not key.endswith("running_var")
        )
        print(f"Learnable parameter tensors total: {parameter_count:,}")
    else:
        print("Could not identify state dictionary")

    print("Checkpoint size (MB):", round(path.stat().st_size / (1024**2), 3))

    if isinstance(obj, dict):
        print("Model name:", obj.get("model_name", obj.get("architecture", "not recorded")))
        print("Class names:", obj.get("class_names", obj.get("class_mapping", "not recorded")))
        print("Best epoch:", obj.get("epoch", obj.get("best_epoch", "not recorded")))

print("\nInspection complete. No files were modified.")

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

BASE = PROJECT_ROOT / "data" / "processed" / "deep_learning"

paths_to_check = [
    BASE / "lightweight_multifeature_cnn",
    BASE / "combined_icbhi_fraiwan_hf_lung_v1"
       / "experiment_c2_icbhi_fraiwan_high_capacity",
]

for folder in paths_to_check:
    print("\n" + "=" * 85)
    print("FOLDER:", folder.relative_to(PROJECT_ROOT))

    if not folder.exists():
        print("Folder not found")
        continue

    for path in sorted(folder.iterdir()):
        if not path.is_file():
            continue

        print(f"\nFile: {path.name}")

        if path.suffix == ".json":
            try:
                print(json.dumps(json.loads(path.read_text()), indent=2)[:5000])
            except Exception as exc:
                print("Could not read JSON:", repr(exc))

        elif path.suffix == ".csv" and (
            "config" in path.name.lower()
            or "manifest" in path.name.lower()
            or "history" in path.name.lower()
        ):
            try:
                df = pd.read_csv(path)
                print("Shape:", df.shape)
                print("Columns:", df.columns.tolist())
                display(df.head(3))
            except Exception as exc:
                print("Could not read CSV:", repr(exc))

        elif path.suffix == ".npz":
            try:
                with np.load(path) as data:
                    print({
                        key: {
                            "shape": data[key].shape,
                            "dtype": str(data[key].dtype),
                            "finite": bool(np.isfinite(data[key]).all()),
                        }
                        for key in data.files
                    })
            except Exception as exc:
                print("Could not inspect NPZ:", repr(exc))

print("\nInspection only: no files modified.")

In [ ]:

from pathlib import Path
import json
import pandas as pd
import torch

BASE = PROJECT_ROOT / "data" / "processed" / "deep_learning"

print("=== SAVED MODEL AND METRIC ARTIFACTS ===")

for path in sorted(BASE.rglob("*")):
    if not path.is_file():
        continue

    name = path.name.lower()
    relevant = (
        path.suffix in {".json", ".csv"}
        and any(term in name for term in [
            "metric", "result", "summary", "history",
            "comparison", "config", "architecture"
        ])
    )

    if relevant:
        print(f"\n--- {path.relative_to(BASE)} ---")
        try:
            if path.suffix == ".json":
                data = json.loads(path.read_text())
                print(json.dumps(data, indent=2, default=str)[:3500])
            else:
                df = pd.read_csv(path)
                print("Shape:", df.shape)
                print(df.head(8).to_string(index=False))
        except Exception as e:
            print("Could not read:", repr(e))

print("\n=== MODEL CHECKPOINT INVENTORY ===")

for path in sorted(BASE.rglob("*.pt")):
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(obj, dict):
            state = obj.get("model_state_dict", obj)
            if isinstance(state, dict):
                tensors = [
                    v for v in state.values()
                    if isinstance(v, torch.Tensor)
                ]
                params = sum(
                    v.numel() for v in tensors
                    if v.is_floating_point()
                )
                metadata = {
                    k: obj[k] for k in [
                        "model_name", "epoch", "best_epoch",
                        "num_classes", "class_names",
                        "best_validation_macro_f1_supported"
                    ] if k in obj
                }
                print(
                    f"{path.relative_to(BASE)} | "
                    f"tensor_count={len(tensors)} | "
                    f"parameter_tensors_total={params:,} | "
                    f"metadata={metadata}"
                )
    except Exception as e:
        print(f"{path.relative_to(BASE)} | inspection failed: {e}")

print("\nAudit only: no files modified.")


In [ ]:

from pathlib import Path

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Root exists:", PROJECT_ROOT.exists())
print("C2 directory:", C2)
print("C2 exists:", C2.exists())

print("\n--- Classical ML objects ---")
for name in ["recording_data", "selected_models", "test_summary"]:
    obj = globals().get(name)
    print(f"{name}:",
          "MISSING" if obj is None else type(obj).__name__)

if "recording_data" in globals():
    for split, item in recording_data.items():
        print(
            split,
            "| X:", item["X"].shape,
            "| labels:", len(item["y"]),
            "| patients:", len(set(item["groups"]))
            if "groups" in item else "not available"
        )

checkpoint_path = (
    C2 / "c2_compact_multifeature_best.pt"
)
print("\nCNN checkpoint exists:", checkpoint_path.is_file())


In [ ]:

import numpy as np
import pandas as pd

print("=== CLASS DISTRIBUTION ===")

for split, item in recording_data.items():
    labels = pd.Series(item["y"]).astype(str)
    print(f"\n{split}: {len(labels)} recordings")
    print(labels.value_counts().to_string())

print("\n=== SELECTED CLASSICAL MODELS ===")

for name, model in selected_models.items():
    print(f"{name}: {type(model).__name__}")

print("\n=== EXISTING TEST SUMMARY ===")

if isinstance(test_summary, pd.DataFrame):
    print("Shape:", test_summary.shape)
    display(test_summary.head(15))
else:
    print("test_summary is:", type(test_summary).__name__)


In [ ]:

print("=== VALIDATION / MODEL-SELECTION OBJECTS ===")

for name, obj in globals().copy().items():
    if any(term in name.lower() for term in
           ["validation", "val_summary", "selection_summary",
            "best_models", "selected_models"]):
        if isinstance(obj, pd.DataFrame):
            print(f"\n{name}: DataFrame {obj.shape}")
            display(obj.head(10))
        elif isinstance(obj, dict):
            print(f"\n{name}: dict with {len(obj)} entries")
            print(list(obj.keys())[:20])

print("\n=== TEST SUMMARY BY TASK ===")
display(
    test_summary.groupby(["task", "test_dataset"])
    .size()
    .rename("number_of_model_results")
    .reset_index()
)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import pandas as pd
import numpy as np

validation_rows = []

y_val_original = np.asarray(recording_data["validation"]["y"]).astype(str)

for (task, model_name), pred in validation_predictions.items():
    pred = np.asarray(pred).astype(str)

    if task == "8_class":
        target_classes = [
            "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
            "Healthy", "LRTI", "Pneumonia", "URTI"
        ]
    else:
        target_classes = [
            "COPD", "Healthy", "Pneumonia",
            "Bronchiectasis", "URTI"
        ]

    # The stored validation predictions may already be filtered
    # to the target task. Verify lengths before computing metrics.
    if len(pred) != len(y_val_original):
        print(
            f"SKIP {task} / {model_name}: "
            f"{len(pred)} predictions vs "
            f"{len(y_val_original)} validation labels"
        )
        continue

    if task == "5_class":
        keep = np.isin(y_val_original, target_classes)
        y_true = y_val_original[keep]
        y_pred = pred[keep]
    else:
        y_true = y_val_original
        y_pred = pred

    supported = sorted(set(y_true) & set(target_classes))

    validation_rows.append({
        "task": task,
        "model": model_name,
        "n_validation": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_supported": balanced_accuracy_score(
            y_true, y_pred
        ),
        "macro_f1_all_target_classes": f1_score(
            y_true, y_pred,
            labels=target_classes,
            average="macro",
            zero_division=0,
        ),
        "macro_f1_supported_classes": f1_score(
            y_true, y_pred,
            labels=supported,
            average="macro",
            zero_division=0,
        ),
    })

validation_metrics = pd.DataFrame(validation_rows)

if validation_metrics.empty:
    print(
        "No metrics computed. Prediction lengths differ from the "
        "full validation set; inspect how validation_predictions "
        "were generated before proceeding."
    )
else:
    display(
        validation_metrics.sort_values(
            ["task", "macro_f1_all_target_classes"],
            ascending=[True, False],
        ).groupby("task", sort=False).head(5)
        .reset_index(drop=True)
    )


In [ ]:

import numpy as np
import pandas as pd

five_classes = [
    "COPD", "Healthy", "Pneumonia",
    "Bronchiectasis", "URTI"
]

y_val = np.asarray(recording_data["validation"]["y"]).astype(str)
mask_5 = np.isin(y_val, five_classes)

print("Total validation recordings:", len(y_val))
print("Recordings belonging to 5-class task:", int(mask_5.sum()))
print("Excluded recordings:", int((~mask_5).sum()))

print("\n5-class validation label counts:")
print(pd.Series(y_val[mask_5]).value_counts().to_string())

print("\nExcluded validation label counts:")
print(pd.Series(y_val[~mask_5]).value_counts().to_string())

print("\nPrediction lengths for 5-class models:")
for key, pred in validation_predictions.items():
    if key[0] == "5_class":
        print(key[1], "->", len(pred))


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import numpy as np
import pandas as pd

classes_8 = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]

classes_5 = [
    "COPD", "Healthy", "Pneumonia",
    "Bronchiectasis", "URTI"
]

y_val = np.asarray(recording_data["validation"]["y"]).astype(str)
validation_rows = []

for (task, model_name), pred in validation_predictions.items():
    pred = np.asarray(pred).astype(str)

    if task == "8_class":
        target_classes = classes_8
        y_true = y_val
    elif task == "5_class":
        target_classes = classes_5
        mask = np.isin(y_val, target_classes)
        y_true = y_val[mask]
    else:
        continue

    if len(pred) != len(y_true):
        print(
            f"Skipping {task} / {model_name}: "
            f"{len(pred)} predictions vs {len(y_true)} labels"
        )
        continue

    supported = [
        cls for cls in target_classes
        if np.any(y_true == cls)
    ]

    validation_rows.append({
        "task": task,
        "model": model_name,
        "n_validation": len(y_true),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy_supported": balanced_accuracy_score(
            y_true, pred
        ),
        "macro_f1_all_target_classes": f1_score(
            y_true, pred,
            labels=target_classes,
            average="macro",
            zero_division=0,
        ),
        "macro_f1_supported_classes": f1_score(
            y_true, pred,
            labels=supported,
            average="macro",
            zero_division=0,
        ),
    })

validation_metrics = pd.DataFrame(validation_rows)

for task in ["8_class", "5_class"]:
    subset = validation_metrics[
        validation_metrics["task"] == task
    ].sort_values(
        "macro_f1_all_target_classes",
        ascending=False
    )

    print(f"\n=== {task.upper()} — TOP 5 VALIDATION MODELS ===")
    display(subset.head(5).reset_index(drop=True))


In [ ]:

# Validation-selected classical candidates
chosen = {
    "8_class": "SVM_RBF_C=1.0",
    "5_class": "SVM_RBF_C=1.0",
}

print("=== VALIDATION-SELECTED CLASSICAL MODELS: HELD-OUT TEST RESULTS ===")

for task, model_name in chosen.items():
    rows = test_summary[
        (test_summary["task"] == task) &
        (test_summary["model"] == model_name)
    ].copy()

    print(f"\n{task} — {model_name}")
    display(
        rows[
            [
                "test_dataset",
                "n_test_recordings",
                "n_supported_classes",
                "accuracy",
                "balanced_accuracy_supported_classes",
                "macro_f1_all_target_classes",
                "macro_f1_supported_classes",
                "weighted_f1",
            ]
        ].reset_index(drop=True)
    )

print("\n=== EXISTING C2 COMPACT CNN SUMMARY FILES ===")
from pathlib import Path

for p in C2.rglob("*summary*"):
    if p.is_file() and p.suffix.lower() in {".csv", ".json"}:
        print(p.relative_to(PROJECT_ROOT))


In [ ]:

from pathlib import Path

search_roots = [
    PROJECT_ROOT / "data" / "processed" / "deep_learning",
    PROJECT_ROOT / "reports",
]

keywords = (
    "c2_compact",
    "compact_multifeature",
    "evaluation",
    "test_summary",
    "prediction",
    "experiment_c2",
)

print("=== EXISTING CNN CHECKPOINTS AND EVALUATION ARTIFACTS ===")

for root in search_roots:
    if not root.exists():
        print("Missing search root:", root)
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        if any(word in name for word in keywords):
            if path.suffix.lower() in {
                ".pt", ".pth", ".csv", ".json", ".npy", ".npz"
            }:
                print(path.relative_to(PROJECT_ROOT))


In [ ]:

from pathlib import Path

c2_root = C2

print("=== C2 DIRECTORY CONTENTS ===")

for path in sorted(c2_root.rglob("*")):
    if path.is_file():
        print(
            f"{path.relative_to(c2_root)}"
            f"  [{path.stat().st_size:,} bytes]"
        )

print("\n=== C1 AND EXPERIMENT B EVALUATION SUMMARIES ===")

base = PROJECT_ROOT / "data" / "processed" / "deep_learning"

for path in sorted(base.rglob("*test_summary*")):
    if path.is_file() and path.suffix.lower() in {".csv", ".json"}:
        print(path.relative_to(PROJECT_ROOT))


In [ ]:

import json
from pathlib import Path

c2_root = C2

print("=== CNN TRAINING HISTORY ===")
history_path = c2_root / "c2_compact_multifeature_training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

if isinstance(history, dict):
    print("Top-level keys:", list(history.keys()))
    for key, value in history.items():
        if isinstance(value, (str, int, float, bool, type(None))):
            print(f"{key}: {value}")
        elif isinstance(value, list):
            print(f"{key}: list, length={len(value)}")
            if value and isinstance(value[-1], dict):
                print("Last entry:", value[-1])
        elif isinstance(value, dict):
            print(f"{key}: dict keys={list(value.keys())[:20]}")

print("\n=== MULTI-FEATURE NORMALIZATION CONFIG ===")
config_path = (
    c2_root
    / "features_multifeature_v2_normalized"
    / "normalization_config.json"
)

if config_path.exists():
    with open(config_path, "r") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Not found:", config_path)

print("\n=== CHECKPOINT SIZE ===")
checkpoint_path = c2_root / "c2_compact_multifeature_best.pt"
print(f"{checkpoint_path.name}: {checkpoint_path.stat().st_size:,} bytes")


In [ ]:

import json
import torch
from pathlib import Path

history_path = C2 / "c2_compact_multifeature_training_history.json"
checkpoint_path = C2 / "c2_compact_multifeature_best.pt"

print("=== HISTORY FILE ===")
print("Bytes:", history_path.stat().st_size)

with open(history_path, "r") as f:
    history = json.load(f)

print("Python type:", type(history).__name__)
print("History value:", repr(history)[:3000])

print("\n=== CHECKPOINT CONTENTS ===")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} keys")
            print("Keys:", list(value.keys())[:30])
        elif isinstance(value, (str, int, float, bool, type(None))):
            print(f"{key}: {value}")
        elif torch.is_tensor(value):
            print(f"{key}: tensor shape={tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__}")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

feature_root = C2 / "features_multifeature_v2_normalized"

files = {
    "train": feature_root / "train_features_normalized.npz",
    "validation": feature_root / "validation_features_normalized.npz",
    "icbhi_test": feature_root / "icbhi_test_features_normalized.npz",
    "fraiwan_test": feature_root / "fraiwan_test_features_normalized.npz",
}

for split, path in files.items():
    print(f"\n=== {split.upper()} ===")

    if not path.exists():
        print("MISSING:", path)
        continue

    with np.load(path, allow_pickle=False) as data:
        print("Array keys:", data.files)
        for key in data.files:
            arr = data[key]
            print(
                f"{key}: shape={arr.shape}, dtype={arr.dtype}, "
                f"finite={np.isfinite(arr).all()}"
            )

metadata_files = {
    "train": C2 / "features_multifeature_v2" / "train_metadata.csv",
    "validation": C2 / "features_multifeature_v2" / "validation_metadata.csv",
    "icbhi_test": C2 / "features_multifeature_v2" / "icbhi_test_metadata.csv",
    "fraiwan_test": C2 / "features_multifeature_v2" / "fraiwan_test_metadata.csv",
}

print("\n=== FEATURE METADATA ===")
for split, path in metadata_files.items():
    print(f"\n{split}: {path.exists()}")
    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(2))


In [ ]:

import torch
from pathlib import Path

checkpoint_path = C2 / "c2_compact_multifeature_best.pt"

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint path:", checkpoint_path)

# Check whether the CNN class is already defined in this notebook session.
model_class = globals().get("CompactMultiFeatureCNN")

print("\nCNN class available:", model_class is not None)
if model_class is not None:
    print("Class:", model_class)
    print("Module:", getattr(model_class, "__module__", "unknown"))

# Inspect the saved checkpoint without training anything.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint architecture:", checkpoint.get("architecture"))
print("Saved class names:", checkpoint.get("class_names"))
print("Saved sample rate:", checkpoint.get("sample_rate"))
print("Saved duration:", checkpoint.get("duration_seconds"), "seconds")
print("Saved best epoch:", checkpoint.get("best_epoch"))
print(
    "Best validation supported-class macro-F1:",
    checkpoint.get("best_validation_macro_f1_supported_classes")
)
print("State-dict tensors:", len(checkpoint["model_state_dict"]))
print("\nFirst state-dict entries:")
for key, tensor in list(checkpoint["model_state_dict"].items())[:12]:
    print(f"{key}: {tuple(tensor.shape)}")


In [ ]:

import torch

# Reuse the existing CNN architecture and saved checkpoint.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_checkpoint = torch.load(
    C2 / "c2_compact_multifeature_best.pt",
    map_location=device,
    weights_only=False
)

cnn_class_names = cnn_checkpoint["class_names"]

# Instantiate the existing architecture.
cnn_model = CompactMultiFeatureCNN()

# Load the saved weights.
load_result = cnn_model.load_state_dict(
    cnn_checkpoint["model_state_dict"],
    strict=True
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

print("Device:", device)
print("Architecture:", cnn_checkpoint["architecture"])
print("Weights loaded strictly:", load_result)
print("Model is in evaluation mode:", not cnn_model.training)
print("Class order:", cnn_class_names)
print("Best checkpoint epoch:", cnn_checkpoint["best_epoch"])
print(
    "Best validation supported-class macro-F1:",
    round(
        cnn_checkpoint["best_validation_macro_f1_supported_classes"],
        6
    )
)
print("Trainable parameters:", cnn_checkpoint["trainable_parameters"])


In [ ]:

import inspect
import numpy as np
import pandas as pd

print("=== CNN FORWARD METHOD ===")
print(inspect.getsource(CompactMultiFeatureCNN.forward))

print("\n=== MODEL INPUT SHAPES ===")
for name, module in cnn_model.named_children():
    print(f"{name}: {module.__class__.__name__}")

print("\n=== CLASS ORDER ===")
print(cnn_class_names)

print("\n=== LABEL ALIGNMENT CHECK ===")
for split in ["train", "validation", "icbhi_test", "fraiwan_test"]:
    metadata_path = C2 / "features_multifeature_v2" / f"{split}_metadata.csv"
    feature_path = C2 / "features_multifeature_v2_normalized" / f"{split}_features_normalized.npz"

    metadata = pd.read_csv(metadata_path)

    with np.load(feature_path, allow_pickle=False) as features:
        print(f"\n{split}:")
        print("Metadata rows:", len(metadata))
        print("Diagnosis counts:")
        print(metadata["diagnosis"].value_counts().to_dict())

        for key in ["logmel", "mfcc", "chroma"]:
            print(f"{key}:", features[key].shape)

        assert len(metadata) == len(features["logmel"]), (
            f"Row mismatch in {split}"
        )
        assert all(
            len(metadata) == len(features[key])
            for key in ["logmel", "mfcc", "chroma"]
        ), f"Feature/metadata row mismatch in {split}"

print("\nAll feature and metadata row counts align.")


In [ ]:

from pathlib import Path
import torch
import numpy as np
import pandas as pd

# Restore the project paths.
PROJECT_ROOT = Path.home() / "Sem 5" / "Respiratory-Disease-Classification"

C2 = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c2_icbhi_fraiwan_high_capacity"
)

feature_root = C2 / "features_multifeature_v2_normalized"
metadata_root = C2 / "features_multifeature_v2"
checkpoint_path = C2 / "c2_compact_multifeature_best.pt"

print("Project exists:", PROJECT_ROOT.exists())
print("C2 directory exists:", C2.exists())
print("Checkpoint exists:", checkpoint_path.exists())
print("Normalized features exist:", feature_root.exists())
print("Metadata directory exists:", metadata_root.exists())
print("PyTorch device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

print("CNN class available:", "CompactMultiFeatureCNN" in globals())
print("Feature branch class available:", "CompactFeatureBranch" in globals())

print("Checkpoint exists:", checkpoint_path.exists())

if "CompactMultiFeatureCNN" in globals():
    print("CNN class:", CompactMultiFeatureCNN)


In [ ]:

import ast

search_roots = [
    PROJECT_ROOT / "Notebooks 2",
    PROJECT_ROOT / "src",
    PROJECT_ROOT,
]

matches = []
seen_files = set()

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path in seen_files:
            continue
        seen_files.add(path)

        if path.suffix not in {".ipynb", ".py"}:
            continue

        # Avoid scanning virtual environments and Git internals.
        if any(part in {".venv", ".git", "__pycache__"} for part in path.parts):
            continue

        try:
            if path.suffix == ".ipynb":
                import json
                notebook = json.loads(path.read_text(encoding="utf-8"))
                contents = "\n".join(
                    "".join(cell.get("source", []))
                    for cell in notebook.get("cells", [])
                    if cell.get("cell_type") == "code"
                )
            else:
                contents = path.read_text(encoding="utf-8")

            if (
                "class CompactMultiFeatureCNN" in contents
                or "class CompactFeatureBranch" in contents
            ):
                matches.append(path)

        except Exception:
            continue

print("Files containing the original CNN class definitions:")
for path in matches:
    print(path)

if not matches:
    print(
        "\nNo source definition found in the scanned locations. "
        "We will inspect the notebook history or other project files next."
    )


In [ ]:

import json
from pathlib import Path

source_notebook = (
    PROJECT_ROOT
    / "Notebooks 2"
    / "08_Combined_ICBHI_Fraiwan_HighCapacity_CNN.ipynb"
)

with open(source_notebook, "r", encoding="utf-8") as f:
    source_nb = json.load(f)

for i, cell in enumerate(source_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if (
        "class CompactFeatureBranch" in source
        or "class CompactMultiFeatureCNN" in source
    ):
        print(f"\n{'=' * 25} SOURCE CELL {i} {'=' * 25}")
        print(source)


In [ ]:

from pathlib import Path
import torch
import numpy as np

PROJECT_ROOT = Path.home() / "Sem 5" / "Respiratory-Disease-Classification"

C2 = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c2_icbhi_fraiwan_high_capacity"
)

checkpoint_path = C2 / "c2_compact_multifeature_best.pt"
feature_root = C2 / "features_multifeature_v2_normalized"
metadata_root = C2 / "features_multifeature_v2"

assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

device = torch.device("cpu")
cnn_checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

cnn_class_names = cnn_checkpoint["class_names"]
cnn_model = CompactMultiFeatureCNN(
    num_classes=len(cnn_class_names)
).to(device)

load_result = cnn_model.load_state_dict(
    cnn_checkpoint["model_state_dict"],
    strict=True,
)
cnn_model.eval()

parameter_count = sum(
    p.numel() for p in cnn_model.parameters() if p.requires_grad
)

with np.load(
    feature_root / "validation_features_normalized.npz",
    allow_pickle=False,
) as features:
    sample_logmel = torch.from_numpy(features["logmel"][:2].copy()).float()
    sample_mfcc = torch.from_numpy(features["mfcc"][:2].copy()).float()
    sample_chroma = torch.from_numpy(features["chroma"][:2].copy()).float()

with torch.no_grad():
    sample_logits = cnn_model(sample_logmel, sample_mfcc, sample_chroma)

print("Checkpoint loaded:", load_result)
print("Architecture:", cnn_checkpoint["architecture"])
print("Trainable parameters:", parameter_count)
print("Output shape:", tuple(sample_logits.shape))
print("Outputs finite:", torch.isfinite(sample_logits).all().item())
print("Evaluation mode:", not cnn_model.training)

assert parameter_count == 19588
assert sample_logits.shape == (2, 8)
assert torch.isfinite(sample_logits).all()

print("\nCheckpoint restored successfully; no training performed.")


In [ ]:

import numpy as np
import pandas as pd
import torch

split_names = ["validation", "icbhi_test", "fraiwan_test"]
cnn_predictions = {}
cnn_model.eval()

with torch.no_grad():
    for split in split_names:
        metadata = pd.read_csv(
            metadata_root / f"{split}_metadata.csv"
        )

        with np.load(
            feature_root / f"{split}_features_normalized.npz",
            allow_pickle=False,
        ) as features:
            x_logmel = torch.from_numpy(features["logmel"].copy()).float()
            x_mfcc = torch.from_numpy(features["mfcc"].copy()).float()
            x_chroma = torch.from_numpy(features["chroma"].copy()).float()

        assert len(metadata) == len(x_logmel) == len(x_mfcc) == len(x_chroma)

        probabilities = []
        for start in range(0, len(metadata), 32):
            end = min(start + 32, len(metadata))

            logits = cnn_model(
                x_logmel[start:end],
                x_mfcc[start:end],
                x_chroma[start:end],
            )
            probabilities.append(
                torch.softmax(logits, dim=1).cpu().numpy()
            )

        probabilities = np.concatenate(probabilities, axis=0)
        assert probabilities.shape == (len(metadata), len(cnn_class_names))

        result = metadata.copy()
        result["true_label"] = result["diagnosis"].astype(str)
        result["predicted_8class"] = [
            cnn_class_names[i] for i in probabilities.argmax(axis=1)
        ]

        for i, class_name in enumerate(cnn_class_names):
            result[f"prob_{class_name}"] = probabilities[:, i]

        cnn_predictions[split] = result

        print(f"\n{split.upper()}")
        print("Recordings:", len(result))
        print("Probability shape:", probabilities.shape)
        print("Predicted class counts:")
        print(result["predicted_8class"].value_counts().to_dict())

print("\nInference complete. No training performed.")


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

classes_8 = list(cnn_class_names)
classes_5 = ["COPD", "Healthy", "Pneumonia", "Bronchiectasis", "URTI"]

evaluation_rows = []
cnn_detailed_reports = {}

for split, result in cnn_predictions.items():
    y_all = result["true_label"].astype(str)

    for task, target_classes in [
        ("8-class", classes_8),
        ("5-class", classes_5),
    ]:
        mask = y_all.isin(target_classes)
        subset = result.loc[mask].copy()

        if subset.empty:
            continue

        y_true = subset["true_label"].to_numpy()

        if task == "8-class":
            y_pred = subset["predicted_8class"].to_numpy()
        else:
            # Conditional five-class evaluation:
            # restrict output to the five eligible diagnosis classes.
            prob_cols = [f"prob_{c}" for c in target_classes]
            restricted_probs = subset[prob_cols].to_numpy()
            y_pred = np.asarray(target_classes)[restricted_probs.argmax(axis=1)]

        present_classes = [
            c for c in target_classes if c in set(y_true)
        ]

        evaluation_rows.append({
            "task": task,
            "split": split,
            "n_recordings": len(subset),
            "n_supported_classes": len(present_classes),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy_supported_classes": balanced_accuracy_score(
                y_true, y_pred
            ),
            "macro_f1_all_target_classes": f1_score(
                y_true, y_pred,
                labels=target_classes,
                average="macro",
                zero_division=0,
            ),
            "macro_f1_supported_classes": f1_score(
                y_true, y_pred,
                labels=present_classes,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_true, y_pred,
                labels=target_classes,
                average="weighted",
                zero_division=0,
            ),
        })

        cnn_detailed_reports[(split, task)] = {
            "y_true": y_true,
            "y_pred": y_pred,
            "labels": target_classes,
            "present_classes": present_classes,
        }

cnn_evaluation_summary = pd.DataFrame(evaluation_rows)

print("========== C2 CNN EVALUATION SUMMARY ==========")
display(
    cnn_evaluation_summary
    .sort_values(["task", "split"])
    .round(4)
    .reset_index(drop=True)
)

print("\nEvaluation completed. No training or model selection performed.")


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for split in ["validation", "icbhi_test", "fraiwan_test"]:
    result = cnn_predictions[split]

    # 8-class confusion matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(
        result["true_label"],
        result["predicted_8class"],
        labels=classes_8,
        display_labels=classes_8,
        xticks_rotation=45,
        cmap="Blues",
        values_format="d",
        ax=ax,
    )
    ax.set_title(f"C2 Compact CNN — {split} — 8 classes")
    plt.tight_layout()
    plt.show()

    # 5-class confusion matrix, restricted to eligible true labels
    subset = result[result["true_label"].isin(classes_5)].copy()

    if len(subset):
        prob_cols = [f"prob_{c}" for c in classes_5]
        pred_5 = np.asarray(classes_5)[
            subset[prob_cols].to_numpy().argmax(axis=1)
        ]

        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(
            subset["true_label"],
            pred_5,
            labels=classes_5,
            display_labels=classes_5,
            xticks_rotation=45,
            cmap="Blues",
            values_format="d",
            ax=ax,
        )
        ax.set_title(f"C2 Compact CNN — {split} — 5 classes")
        plt.tight_layout()
        plt.show()


In [ ]:

from pathlib import Path
import pandas as pd

# Find existing evaluation artifacts without opening or modifying model files.
search_root = PROJECT_ROOT / "data" / "processed" / "deep_learning"

keywords = (
    "summary", "metric", "result", "evaluation",
    "prediction", "report", "comparison"
)

artifact_files = []

for path in search_root.rglob("*"):
    if not path.is_file():
        continue

    if any(part in {".venv", "__pycache__"} for part in path.parts):
        continue

    name = path.name.lower()

    if path.suffix.lower() in {".csv", ".json", ".xlsx"} and any(
        word in name for word in keywords
    ):
        artifact_files.append(path)

print("=== EXISTING EVALUATION ARTIFACTS ===")
print("Files found:", len(artifact_files))

for path in sorted(artifact_files):
    try:
        size_kb = path.stat().st_size / 1024
        print(f"{size_kb:9.1f} KB | {path.relative_to(PROJECT_ROOT)}")
    except OSError:
        print(path)

print("\n=== CURRENT CLASSICAL ML SUMMARY ===")
if "test_summary" in globals() and isinstance(test_summary, pd.DataFrame):
    print("Shape:", test_summary.shape)
    print("Columns:", test_summary.columns.tolist())
    display(test_summary.head(8))
else:
    print("test_summary is not currently loaded in this kernel.")

print("\n=== CURRENT CNN SUMMARY ===")
if "cnn_evaluation_summary" in globals() and isinstance(
    cnn_evaluation_summary, pd.DataFrame
):
    print("Shape:", cnn_evaluation_summary.shape)
    display(cnn_evaluation_summary)
else:
    print("CNN evaluation summary is not currently loaded.")


In [ ]:

from pathlib import Path
import json
import pandas as pd

search_root = PROJECT_ROOT / "data" / "processed" / "deep_learning"

summary_files = []

for path in search_root.rglob("*"):
    if not path.is_file():
        continue
    if any(part in {".venv", ".git", "__pycache__", ".ipynb_checkpoints"} for part in path.parts):
        continue
    if path.suffix.lower() not in {".csv", ".json"}:
        continue

    name = path.name.lower()
    if any(word in name for word in ["summary", "metrics", "comparison", "experiment_summary"]):
        summary_files.append(path)

print("Summary files found:", len(summary_files))

for path in sorted(summary_files):
    print(f"\n{'=' * 80}")
    print(path.relative_to(PROJECT_ROOT))

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(10))
        else:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict):
                print(json.dumps(data, indent=2, default=str)[:5000])
            elif isinstance(data, list):
                print(json.dumps(data[:10], indent=2, default=str)[:5000])
            else:
                print(str(data)[:2000])
    except Exception as exc:
        print("Could not read:", repr(exc))


In [ ]:

from pathlib import Path
import pandas as pd

search_root = PROJECT_ROOT / "data" / "processed"

terms = [
    "validation",
    "model_comparison",
    "classical",
    "traditional",
    "feature_based",
    "ml_comparison",
]

candidate_files = []

for path in search_root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".csv", ".json"}:
        continue

    if any(
        part in {".venv", ".git", "__pycache__", ".ipynb_checkpoints"}
        for part in path.parts
    ):
        continue

    name = path.name.lower()
    parent = str(path.parent).lower()

    if any(term in name or term in parent for term in terms):
        candidate_files.append(path)

print("Potential saved ML/validation reports:", len(candidate_files))

for path in sorted(candidate_files):
    print(path.relative_to(PROJECT_ROOT))
    if path.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(path, nrows=3)
            print("  Columns:", df.columns.tolist())
        except Exception as exc:
            print("  Read error:", repr(exc))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

SEARCH_ROOTS = [
    PROJECT_ROOT / "data" / "processed",
]

extensions = {
    ".pt", ".pth", ".keras", ".h5", ".pkl", ".joblib",
    ".json", ".csv", ".xlsx"
}

keywords = (
    "model", "metric", "summary", "comparison", "result",
    "evaluation", "report", "prediction", "checkpoint",
    "experiment", "patient", "imbalance", "cnn", "svm",
    "forest", "xgboost", "logistic"
)

found_files = []

for root in SEARCH_ROOTS:
    if not root.exists():
        print(f"Directory not found: {root}")
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in extensions:
            continue

        name = path.name.lower()
        if any(word in name for word in keywords):
            try:
                size_kb = round(path.stat().st_size / 1024, 1)
            except OSError:
                size_kb = None

            found_files.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "type": path.suffix.lower(),
                "size_kb": size_kb,
            })

inventory = (
    pd.DataFrame(found_files)
    .drop_duplicates(subset="file")
    .sort_values(["type", "file"])
    .reset_index(drop=True)
    if found_files
    else pd.DataFrame(columns=["file", "type", "size_kb"])
)

print(f"Project root exists: {PROJECT_ROOT.exists()}")
print(f"Result/model files found: {len(inventory)}")
display(inventory)


In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
DATA_ROOT = PROJECT_ROOT / "data" / "processed"

checkpoint_exts = {".pt", ".pth", ".keras", ".h5", ".pkl", ".joblib"}
result_exts = {".csv", ".json", ".xlsx"}

checkpoints = []
result_files = []

for path in DATA_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if ".ipynb_checkpoints" in path.parts:
        continue

    rel_path = str(path.relative_to(PROJECT_ROOT))
    suffix = path.suffix.lower()

    if suffix in checkpoint_exts:
        checkpoints.append({
            "model_file": rel_path,
            "size_kb": round(path.stat().st_size / 1024, 1),
        })

    elif suffix in result_exts:
        name = path.name.lower()
        if any(word in name for word in (
            "metric", "summary", "comparison", "result",
            "evaluation", "report", "prediction", "experiment"
        )):
            result_files.append(path)

checkpoint_inventory = pd.DataFrame(checkpoints)
if not checkpoint_inventory.empty:
    checkpoint_inventory = checkpoint_inventory.sort_values("model_file")

print("=== SAVED MODEL / CHECKPOINT FILES ===")
print(f"Count: {len(checkpoint_inventory)}")
display(checkpoint_inventory)

print("\n=== RESULT FILE CONTENTS ===")
result_inventory = []

for path in sorted(set(result_files)):
    rel_path = str(path.relative_to(PROJECT_ROOT))
    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=5)
            result_inventory.append({
                "file": rel_path,
                "format": "CSV",
                "columns": ", ".join(map(str, df.columns)),
                "sample_rows": len(df),
            })

        elif path.suffix.lower() == ".json":
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)

            keys = list(obj.keys()) if isinstance(obj, dict) else [
                f"JSON list ({len(obj)} items)"
            ]
            result_inventory.append({
                "file": rel_path,
                "format": "JSON",
                "columns": ", ".join(map(str, keys)),
                "sample_rows": "",
            })

        elif path.suffix.lower() == ".xlsx":
            df = pd.read_excel(path, nrows=5)
            result_inventory.append({
                "file": rel_path,
                "format": "Excel",
                "columns": ", ".join(map(str, df.columns)),
                "sample_rows": len(df),
            })

    except Exception as e:
        result_inventory.append({
            "file": rel_path,
            "format": path.suffix.lower(),
            "columns": f"Could not read: {e}",
            "sample_rows": "",
        })

result_inventory = pd.DataFrame(result_inventory)
print(f"Readable result files: {len(result_inventory)}")
display(result_inventory)


In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
DATA_ROOT = PROJECT_ROOT / "data" / "processed"

keywords = (
    "model_comparison",
    "final_classical_ml_comparison",
    "experiment_summary",
    "test_metrics",
    "overall_metrics",
    "evaluation_summary",
    "final_model_comparison",
    "cnn_baseline_experiment_summary",
    "imbalance_comparison",
    "patient_level_comparison",
)

files = []
for path in DATA_ROOT.rglob("*"):
    if not path.is_file() or ".ipynb_checkpoints" in path.parts:
        continue
    if path.suffix.lower() not in {".csv", ".json"}:
        continue
    if any(k in path.name.lower() for k in keywords):
        files.append(path)

print(f"Priority result files found: {len(files)}")

for path in sorted(set(files)):
    print("\n" + "=" * 100)
    print(path.relative_to(PROJECT_ROOT))

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            print(f"Shape: {df.shape}")
            display(df.head(30))

        else:
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)
            print(json.dumps(obj, indent=2, default=str)[:8000])

    except Exception as e:
        print("Could not read:", repr(e))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
DATA_ROOT = PROJECT_ROOT / "data" / "processed"

files_to_check = [
    "classical_ml/classical_ml_model_comparison.csv",
    "classical_ml/final_classical_ml_comparison.csv",
    "deep_learning/cnn_lstm/cnn_model_comparison.csv",
    "deep_learning/multifeature_cnn/final_cnn_experiment_comparison.csv",
]

for relative_path in files_to_check:
    path = DATA_ROOT / relative_path

    print("\n" + "=" * 90)
    print(relative_path)

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        display(df)
    else:
        print("File not found; checking for similarly named files.")

        parent = path.parent
        if parent.exists():
            for candidate in sorted(parent.glob("*.csv")):
                print("  ", candidate.name)

print("\n=== COMBINED-DATASET RESULT FILES ===")

combined_root = DATA_ROOT / "deep_learning" / "combined_icbhi_fraiwan_hf_lung_v1"

if combined_root.exists():
    for path in sorted(combined_root.rglob("*")):
        if not path.is_file() or ".ipynb_checkpoints" in path.parts:
            continue
        if path.suffix.lower() in {".csv", ".json"}:
            print(path.relative_to(PROJECT_ROOT))
else:
    print("Combined-dataset results directory not found.")


In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
BASE = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

summary_files = [
    BASE / "experiment_c1_icbhi_only" / "evaluation" / "c1_test_summary.csv",
    BASE / "experiment_c1_icbhi_only" / "evaluation_c1_normalized" / "summary.csv",
    BASE / "experiment_b_feature_cache" / "experiment_b_v1_strict" / "evaluation" / "experiment_b_test_summary.csv",
]

for path in summary_files:
    print("\n" + "=" * 90)
    print(path.relative_to(PROJECT_ROOT))

    if path.exists():
        try:
            df = pd.read_csv(path)
            display(df)
        except Exception as e:
            print("Could not read CSV:", repr(e))
    else:
        print("Not found")

print("\n=== C2 TRAINING HISTORY / BEST CHECKPOINT METADATA ===")

c2 = BASE / "experiment_c2_icbhi_fraiwan_high_capacity"

for filename in [
    "c2_compact_multifeature_training_history.json",
    "c2_compact_multifeature_best.pt",
]:
    path = c2 / filename
    print("\n", filename)

    if not path.exists():
        print("Not found")
        continue

    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2)[:6000])
    else:
        print(f"Checkpoint exists ({path.stat().st_size / 1024:.1f} KB)")

print("\n=== EXISTING LIGHTWEIGHT CNN METRICS ===")

path = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "lightweight_multifeature_cnn" / "test_metrics.json"
)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Metrics file not found")


In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
DATA_ROOT = PROJECT_ROOT / "data" / "processed"

records = []

metric_aliases = {
    "accuracy", "test_accuracy",
    "balanced_accuracy", "test_balanced_accuracy",
    "balanced_accuracy_supported_classes",
    "macro_f1", "test_macro_f1",
    "macro_f1_all8", "macro_f1_all_8_classes",
    "macro_f1_supported", "macro_f1_supported_classes",
    "weighted_f1", "test_weighted_f1",
    "test_samples", "test_cycles", "test_patients",
    "recordings", "windows",
}

for path in sorted(DATA_ROOT.rglob("*")):
    if not path.is_file() or ".ipynb_checkpoints" in path.parts:
        continue
    if path.suffix.lower() not in {".csv", ".json"}:
        continue

    name = path.name.lower()
    if not any(word in name for word in (
        "summary", "metrics", "comparison", "evaluation",
        "experiment", "overall", "test"
    )):
        continue

    rel = str(path.relative_to(PROJECT_ROOT))

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)

            for _, row in df.iterrows():
                item = {
                    "source_file": rel,
                    "row_name": str(
                        row.get("model",
                        row.get("experiment",
                        row.get("dataset",
                        row.get("test_split", ""))))
                    ),
                }

                for col in df.columns:
                    if str(col).lower() in metric_aliases:
                        value = row[col]
                        item[str(col)] = (
                            value if pd.notna(value) else None
                        )

                if len(item) > 2:
                    records.append(item)

        else:
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)

            items = obj if isinstance(obj, list) else [obj]

            for entry in items:
                if not isinstance(entry, dict):
                    continue

                item = {
                    "source_file": rel,
                    "row_name": str(
                        entry.get("model",
                        entry.get("experiment",
                        entry.get("dataset", "")))
                    ),
                }

                for key, value in entry.items():
                    if str(key).lower() in metric_aliases:
                        item[str(key)] = value

                if len(item) > 2:
                    records.append(item)

    except Exception as e:
        print(f"Skipped unreadable file: {rel} ({e})")

master_inventory = pd.DataFrame(records)

if not master_inventory.empty:
    master_inventory = master_inventory.loc[
        :, ~master_inventory.columns.duplicated()
    ]
    master_inventory = master_inventory.sort_values(
        ["source_file", "row_name"]
    ).reset_index(drop=True)

print("Evaluation rows collected:", len(master_inventory))
print("Distinct source files:", master_inventory["source_file"].nunique()
      if not master_inventory.empty else 0)

display(master_inventory)


In [ ]:

# Build a cleaner view from the existing master_inventory DataFrame.

import pandas as pd

if "master_inventory" not in globals() or master_inventory.empty:
    raise RuntimeError(
        "master_inventory is unavailable. Run the previous inventory cell first."
    )

master_clean = master_inventory.copy()

# Keep rows that contain at least one meaningful performance metric.
metric_cols = [
    c for c in [
        "accuracy", "Accuracy", "test_accuracy",
        "balanced_accuracy", "test_balanced_accuracy",
        "balanced_accuracy_supported_classes",
        "macro_f1", "macro_f1_supported_classes",
        "macro_f1_all_8_classes", "weighted_f1", "test_weighted_f1"
    ]
    if c in master_clean.columns
]

master_clean = master_clean[
    master_clean[metric_cols].notna().any(axis=1)
].copy()

# Merge alternate spellings of accuracy into a single column.
for col in ["Accuracy", "test_accuracy"]:
    if col in master_clean.columns:
        master_clean["accuracy"] = (
            master_clean["accuracy"].combine_first(master_clean[col])
            if "accuracy" in master_clean.columns
            else master_clean[col]
        )

# Identify evaluation level from available sample-count fields.
def infer_level(row):
    path = str(row.get("source_file", "")).lower()

    if "patient_level" in path:
        return "Patient-level"
    if "combined_icbhi_fraiwan" in path:
        return "Recording-level experiment"
    if "test_cycles" in row.index and pd.notna(row.get("test_cycles")):
        return "Cycle-level"
    if "test_samples" in row.index and pd.notna(row.get("test_samples")):
        return "Cycle/sample-level"
    if "classical_ml" in path:
        return "Legacy classical ML; verify split"
    return "Needs verification"

master_clean["evaluation_level"] = master_clean.apply(
    infer_level, axis=1
)

# Remove exact duplicate result rows, but preserve distinct experiments/files.
dedup_cols = [
    c for c in [
        "source_file", "row_name", "evaluation_level",
        "accuracy", "balanced_accuracy",
        "balanced_accuracy_supported_classes",
        "macro_f1", "macro_f1_supported_classes",
        "macro_f1_all_8_classes", "weighted_f1",
        "test_weighted_f1", "test_cycles", "test_samples",
        "recordings", "windows", "test_patients"
    ]
    if c in master_clean.columns
]

master_clean = master_clean.drop_duplicates(
    subset=dedup_cols
).reset_index(drop=True)

print("Rows after metric filtering and deduplication:", len(master_clean))
print("\nEvaluation levels:")
display(
    master_clean["evaluation_level"]
    .value_counts()
    .rename_axis("evaluation_level")
    .reset_index(name="rows")
)

display(master_clean)


In [ ]:

from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

extensions = {".pkl", ".joblib", ".sav", ".model", ".ubj", ".json"}
matches = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if ".ipynb_checkpoints" in path.parts or ".git" in path.parts:
        continue

    if path.suffix.lower() not in extensions:
        continue

    name = path.name.lower()
    relevant = any(word in name for word in [
        "model", "pipeline", "logistic", "svm", "forest",
        "xgb", "xgboost", "classifier", "estimator",
        "predict", "prediction", "artifact"
    ])

    if relevant:
        matches.append({
            "file": str(path.relative_to(PROJECT_ROOT)),
            "extension": path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 1)
        })

import pandas as pd

model_artifact_inventory = pd.DataFrame(matches)

if not model_artifact_inventory.empty:
    model_artifact_inventory = (
        model_artifact_inventory
        .drop_duplicates(subset="file")
        .sort_values("file")
        .reset_index(drop=True)
    )

print("Potential traditional ML artifacts:", len(model_artifact_inventory))
display(model_artifact_inventory)


In [ ]:

from pathlib import Path
import json

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = [
    PROJECT_ROOT / "models" / "sound_phenotype_cnn" / "xgboost"
    / "xgboost_sound_phenotype_model.json",
    PROJECT_ROOT / "reports" / "final_artifacts"
    / "model_package_manifest.json",
]

for path in files:
    print("\n" + "=" * 90)
    print("File:", path.relative_to(PROJECT_ROOT))

    if not path.exists():
        print("File not found. Check the exact filename.")
        continue

    print("Size (KB):", round(path.stat().st_size / 1024, 1))

    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            print("Top-level keys:", list(obj.keys())[:40])

            # Print useful metadata without dumping a potentially huge model.
            for key in [
                "model", "model_name", "model_type", "task",
                "classes", "class_names", "features", "feature_names",
                "n_features", "objective", "learner", "models",
                "artifacts", "files", "version"
            ]:
                if key in obj:
                    value = obj[key]
                    text = repr(value)
                    print(f"{key}: {text[:1800]}")
        else:
            print("JSON type:", type(obj).__name__)
            print("Preview:", repr(obj)[:1800])

    except Exception as e:
        print("Could not parse JSON:", repr(e))


In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

manifest_path = (
    PROJECT_ROOT / "reports" / "final_artifacts"
    / "model_package_manifest.json"
)

if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest_df = pd.DataFrame(manifest)

# Exclude sound phenotype artifacts; they are a separate task.
if "relative_path" in manifest_df.columns:
    disease_df = manifest_df[
        ~manifest_df["relative_path"].astype(str).str.contains(
            "sound_phenotype", case=False, na=False
        )
    ].copy()
else:
    disease_df = manifest_df.copy()

# Keep artifacts that help identify trained models or evaluation outputs.
keywords = (
    "model", "checkpoint", "prediction", "metric", "history",
    "config", "normalization", "report", "confusion"
)

if "artifact" in disease_df.columns:
    relevant = disease_df["artifact"].astype(str).str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
    disease_df = disease_df[relevant]

print("Disease-task package entries:", len(disease_df))
print("\n=== ARTIFACTS BY MODEL ===")

display_cols = [
    c for c in [
        "model_id", "artifact", "relative_path",
        "size_bytes", "status"
    ]
    if c in disease_df.columns
]
display(disease_df[display_cols].sort_values(
    ["model_id", "artifact"], na_position="last"
))

print("\n=== STATUS COUNTS ===")
if "status" in disease_df.columns:
    display(
        disease_df.groupby(["model_id", "status"])
        .size()
        .reset_index(name="artifact_count")
    )


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

prediction_files = [
    "models/saved_models/lightweight_mel_cnn/cnn_test_predictions.csv",
    "models/saved_models/cnn_lstm/cnn_lstm_test_predictions.csv",
    "models/saved_models/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "models/saved_models/regularized_multifeature_cnn/test_predictions.csv",
]

for relative_path in prediction_files:
    path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 90)
    print(relative_path)

    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

    # Show the likely true-label and predicted-label columns.
    label_cols = [
        col for col in df.columns
        if any(term in col.lower() for term in [
            "true", "actual", "target", "label",
            "pred", "patient_id", "recording_id"
        ])
    ]

    print("Label / identifier columns:", label_cols)

    for col in label_cols:
        if any(term in col.lower() for term in ["true", "actual", "target", "label", "pred"]):
            print(f"\n{col} value counts:")
            display(df[col].value_counts(dropna=False).head(15))


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

wanted_names = {
    "cnn_test_predictions.csv",
    "cnn_lstm_test_predictions.csv",
    "multifeature_cnn_test_predictions.csv",
    "test_predictions.csv",
}

matches = []

for path in PROJECT_ROOT.rglob("*.csv"):
    if ".ipynb_checkpoints" in path.parts or ".git" in path.parts:
        continue

    if path.name.lower() in wanted_names:
        matches.append({
            "filename": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "size_kb": round(path.stat().st_size / 1024, 1),
        })

matches_df = pd.DataFrame(matches)

print("Prediction files located:", len(matches_df))
display(
    matches_df.sort_values(["filename", "relative_path"])
    if not matches_df.empty
    else matches_df
)

print("\n=== MANIFEST PATH VALIDATION ===")

manifest_path = (
    PROJECT_ROOT / "reports" / "final_artifacts"
    / "model_package_manifest.json"
)

if manifest_path.exists():
    import json

    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    manifest_df = pd.DataFrame(manifest)

    for _, row in manifest_df.iterrows():
        rel = row.get("relative_path")
        if isinstance(rel, str) and "prediction" in rel.lower():
            actual_path = PROJECT_ROOT / rel
            print(
                f"{'FOUND' if actual_path.exists() else 'MISSING'} | "
                f"{rel}"
            )


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

prediction_files = {
    "Lightweight Mel CNN": (
        "data/processed/deep_learning/cnn_baseline/"
        "cnn_test_predictions.csv"
    ),
    "CNN-LSTM": (
        "data/processed/deep_learning/cnn_lstm/"
        "cnn_lstm_test_predictions.csv"
    ),
    "Multi-feature CNN": (
        "data/processed/deep_learning/multifeature_cnn/"
        "multifeature_cnn_test_predictions.csv"
    ),
    "Lightweight Multi-feature CNN": (
        "data/processed/deep_learning/lightweight_multifeature_cnn/"
        "test_predictions.csv"
    ),
    "Regularized Multi-feature CNN": (
        "data/processed/deep_learning/multifeature_cnn_regularized/"
        "test_predictions.csv"
    ),
}

for model_name, relative_path in prediction_files.items():
    path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 85)
    print("MODEL:", model_name)
    print("PATH:", relative_path)

    if not path.exists():
        print("NOT FOUND at this candidate path")
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

    possible_cols = [
        col for col in df.columns
        if any(term in col.lower() for term in [
            "true", "actual", "target", "label", "pred",
            "patient", "recording", "file", "cycle"
        ])
    ]
    print("Potential label / identifier columns:", possible_cols)


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "Lightweight Mel CNN":
        "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    "CNN-LSTM":
        "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "Multi-feature CNN":
        "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Lightweight Multi-feature CNN":
        "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
}

results = []
sample_keys = {}
class_orders = {}

for model, rel_path in files.items():
    path = PROJECT_ROOT / rel_path
    if not path.exists():
        print(f"SKIPPED — missing file: {model}")
        continue

    df = pd.read_csv(path)

    # Normalize the saved prediction-column names.
    if model == "Lightweight Mel CNN":
        y_true = df["true_label"]
        y_pred = df["predicted_label"]
        keys = df[["patient_id", "recording_id"]].astype(str)
    elif model == "CNN-LSTM":
        y_true = df["disease_label"]
        y_pred = df["predicted_disease_label"]
        keys = df[["patient_id", "recording_id"]].astype(str)
    elif model == "Multi-feature CNN":
        y_true = df["true_label"]
        y_pred = df["predicted_label"]
        # This file has no patient/recording identifiers.
        keys = None
    else:
        y_true = df["true_label"]
        y_pred = df["predicted_label"]
        keys = df[["audio_path"]].astype(str)

    y_true = pd.to_numeric(y_true, errors="coerce")
    y_pred = pd.to_numeric(y_pred, errors="coerce")
    valid = y_true.notna() & y_pred.notna()
    y_true = y_true[valid].astype(int)
    y_pred = y_pred[valid].astype(int)

    # Calculate all-eight-class macro metrics, including absent classes.
    labels = list(range(8))
    results.append({
        "model": model,
        "n_predictions": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_8class": balanced_accuracy_score(
            y_true, y_pred, labels=labels
        ),
        "macro_precision_8class": precision_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_recall_8class": recall_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_f1_8class": f1_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "true_classes_present": y_true.nunique(),
        "predicted_classes_present": y_pred.nunique(),
    })

    sample_keys[model] = (
        keys.loc[valid].reset_index(drop=True)
        if keys is not None else None
    )

    print(f"\n{model}")
    print("Rows:", len(df), "| valid predictions:", len(y_true))
    print("True class counts:")
    print(y_true.value_counts().sort_index().to_dict())
    print("Predicted class counts:")
    print(y_pred.value_counts().sort_index().to_dict())

print("\n" + "=" * 100)
print("SAVED-PREDICTION METRICS — ALL 8 CLASSES")
metrics_df = pd.DataFrame(results)
display(metrics_df.round(4))

print("\n" + "=" * 100)
print("SAMPLE ALIGNMENT CHECK")

for model, keys in sample_keys.items():
    if keys is None:
        print(f"{model}: cannot verify sample alignment from this file alone.")
    else:
        print(
            f"{model}: {len(keys)} rows; "
            f"{len(keys.drop_duplicates())} unique identifier rows."
        )


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "Lightweight Mel CNN":
        "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    "CNN-LSTM":
        "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "Multi-feature CNN":
        "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Lightweight Multi-feature CNN":
        "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
}

results = []
sample_keys = {}
labels = list(range(8))

for model, rel_path in files.items():
    path = PROJECT_ROOT / rel_path

    print("\n" + "=" * 80)
    print("MODEL:", model)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)

    if model == "Lightweight Mel CNN":
        true_col, pred_col = "true_label", "predicted_label"
        key_cols = ["patient_id", "recording_id"]
    elif model == "CNN-LSTM":
        true_col, pred_col = "disease_label", "predicted_disease_label"
        key_cols = ["patient_id", "recording_id"]
    elif model == "Multi-feature CNN":
        true_col, pred_col = "true_label", "predicted_label"
        key_cols = None
    else:
        true_col, pred_col = "true_label", "predicted_label"
        key_cols = ["audio_path"]

    y_true = pd.to_numeric(df[true_col], errors="coerce")
    y_pred = pd.to_numeric(df[pred_col], errors="coerce")
    valid = y_true.notna() & y_pred.notna()

    y_true = y_true.loc[valid].astype(int)
    y_pred = y_pred.loc[valid].astype(int)

    results.append({
        "model": model,
        "n_predictions": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_present_classes":
            balanced_accuracy_score(y_true, y_pred),
        "macro_precision_8class": precision_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_recall_8class": recall_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_f1_8class": f1_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "true_classes_present": y_true.nunique(),
        "predicted_classes_present": y_pred.nunique(),
    })

    if key_cols:
        sample_keys[model] = (
            df.loc[valid, key_cols].astype(str).reset_index(drop=True)
        )
    else:
        sample_keys[model] = None

    print("Valid predictions:", len(y_true))
    print("True class counts:", y_true.value_counts().sort_index().to_dict())
    print("Predicted class counts:", y_pred.value_counts().sort_index().to_dict())

print("\n" + "=" * 80)
print("SAVED-PREDICTION METRICS")
metrics_df = pd.DataFrame(results)
display(metrics_df.round(4))

print("\n" + "=" * 80)
print("SAMPLE ALIGNMENT CHECK")

for model, keys in sample_keys.items():
    if keys is None:
        print(f"{model}: identifiers unavailable in this prediction file.")
    else:
        print(
            f"{model}: {len(keys)} rows, "
            f"{len(keys.drop_duplicates())} unique identifier rows."
        )


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

paths = {
    "Mel CNN": ROOT / "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    "CNN-LSTM": ROOT / "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "Multi-feature CNN": ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Lightweight Multi-feature CNN": ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
}

# Build a normalized sample identifier for files that contain recording IDs.
key_sets = {}

for name in ["Mel CNN", "CNN-LSTM", "Lightweight Multi-feature CNN"]:
    df = pd.read_csv(paths[name])

    if name in ["Mel CNN", "CNN-LSTM"]:
        keys = (
            df["patient_id"].astype(str).str.strip()
            + "|"
            + df["recording_id"].astype(str).str.strip()
        )
    else:
        keys = df["audio_path"].astype(str).str.strip()

    key_sets[name] = set(keys)

print("=== OVERLAP OF IDENTIFIERS ===")
names = list(key_sets)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        intersection = key_sets[a] & key_sets[b]
        union = key_sets[a] | key_sets[b]

        print(
            f"{a} vs {b}: "
            f"{len(intersection)} shared / {len(union)} total unique identifiers; "
            f"Jaccard={len(intersection) / len(union):.4f}"
        )

print("\n=== MULTI-FEATURE CNN FILE METADATA ===")
multi = pd.read_csv(paths["Multi-feature CNN"])
print("Columns:", multi.columns.tolist())
print("Rows:", len(multi))
print("First 10 true/predicted labels:")
display(multi[["true_label", "predicted_label"]].head(10))

print("\n=== POSSIBLE REGULARIZED MODEL FILES ===")
for folder in [
    ROOT / "data/processed/deep_learning",
    ROOT / "models/deep_learning",
]:
    if folder.exists():
        for p in folder.rglob("*test_predictions.csv"):
            if "sound_phenotype" not in str(p).lower():
                print(p.relative_to(ROOT))


In [ ]:
 
from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "Regularized Multi-feature CNN":
        "models/deep_learning/regularized_multifeature_cnn/test_predictions.csv",
    "Experiment A — Weighted CE":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B — Balanced Sampler":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
}

labels = list(range(8))
loaded = {}
results = []

for name, relative_path in files.items():
    path = ROOT / relative_path

    print("\n" + "=" * 85)
    print(name)
    print("Path:", relative_path)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

    true_candidates = [
        c for c in df.columns
        if c.lower() in {"true_label", "y_true", "true_disease", "disease_label"}
    ]
    pred_candidates = [
        c for c in df.columns
        if c.lower() in {"predicted_label", "y_pred", "predicted_disease_label"}
    ]

    if not true_candidates or not pred_candidates:
        print("Could not identify true/predicted label columns automatically.")
        continue

    true_col, pred_col = true_candidates[0], pred_candidates[0]
    y_true = pd.to_numeric(df[true_col], errors="coerce")
    y_pred = pd.to_numeric(df[pred_col], errors="coerce")
    valid = y_true.notna() & y_pred.notna()

    y_true = y_true[valid].astype(int)
    y_pred = y_pred[valid].astype(int)

    results.append({
        "model": name,
        "n_predictions": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_present_classes":
            balanced_accuracy_score(y_true, y_pred),
        "macro_precision_8class": precision_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_recall_8class": recall_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_f1_8class": f1_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "true_classes_present": y_true.nunique(),
        "predicted_classes_present": y_pred.nunique(),
    })

    loaded[name] = {
        "df": df.loc[valid].reset_index(drop=True),
        "true": y_true.reset_index(drop=True),
        "pred": y_pred.reset_index(drop=True),
        "true_col": true_col,
        "pred_col": pred_col,
    }

    print("True class counts:", y_true.value_counts().sort_index().to_dict())
    print("Predicted class counts:", y_pred.value_counts().sort_index().to_dict())

print("\n" + "=" * 85)
print("ADDITIONAL MODEL METRICS")
display(pd.DataFrame(results).round(4))

a_name = "Experiment A — Weighted CE"
b_name = "Experiment B — Balanced Sampler"

if a_name in loaded and b_name in loaded:
    a, b = loaded[a_name], loaded[b_name]

    print("\n=== EXPERIMENT A vs B ALIGNMENT ===")
    print("Same number of rows:", len(a["true"]) == len(b["true"]))
    print("True labels match row by row:", a["true"].equals(b["true"]))

    # Compare identifiers if both files have a likely identifier column.
    id_candidates = ["audio_path", "recording_id", "patient_id", "sample_id"]
    shared_ids = [
        c for c in id_candidates
        if c in a["df"].columns and c in b["df"].columns
    ]

    if shared_ids:
        for col in shared_ids:
            print(
                f"{col} matches row by row:",
                a["df"][col].astype(str).equals(b["df"][col].astype(str))
            )
    else:
        print("No common explicit sample identifier column found.")


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "Regularized Multi-feature CNN":
        "models/deep_learning/regularized_multifeature_cnn/test_predictions.csv",
    "Experiment A — Weighted CE":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B — Balanced Sampler":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
}

rows = []
loaded = {}
labels = list(range(8))

for name, rel_path in files.items():
    path = ROOT / rel_path
    if not path.exists():
        print(f"Missing: {name}")
        continue

    df = pd.read_csv(path)
    y_true = pd.to_numeric(df["true_label"], errors="coerce")
    y_pred = pd.to_numeric(df["predicted_label"], errors="coerce")
    valid = y_true.notna() & y_pred.notna()

    df = df.loc[valid].reset_index(drop=True)
    y_true = y_true.loc[valid].astype(int)
    y_pred = y_pred.loc[valid].astype(int)

    rows.append({
        "model": name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_present_classes":
            balanced_accuracy_score(y_true, y_pred),
        "macro_f1_all_8_classes": f1_score(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        ),
        "macro_f1_supported_classes": f1_score(
            y_true, y_pred,
            labels=sorted(y_true.unique()),
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "predicted_classes": y_pred.nunique(),
    })

    loaded[name] = df

print("=== METRICS ===")
display(pd.DataFrame(rows).round(4))

a_name = "Experiment A — Weighted CE"
b_name = "Experiment B — Balanced Sampler"

if a_name in loaded and b_name in loaded:
    a = loaded[a_name]
    b = loaded[b_name]

    print("\n=== A/B ALIGNMENT ===")
    print("Same row count:", len(a) == len(b))

    for col in ["patient_id", "recording_id", "audio_path", "true_label"]:
        if col in a.columns and col in b.columns:
            print(
                f"{col} identical row by row:",
                a[col].astype(str).reset_index(drop=True).equals(
                    b[col].astype(str).reset_index(drop=True)
                )
            )


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

print("=== NOTEBOOK INVENTORY ===")

notebook_rows = []

for path in sorted(ROOT.rglob("*.ipynb")):
    if any(part in {".venv", ".git", ".ipynb_checkpoints"} for part in path.parts):
        continue

    notebook_rows.append({
        "notebook": str(path.relative_to(ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

notebooks_df = pd.DataFrame(notebook_rows)
display(notebooks_df)

print("\n=== MODEL / EXPERIMENT RESULT SUMMARIES ===")

summary_rows = []
summary_terms = (
    "summary", "comparison", "metrics", "evaluation",
    "results", "history"
)

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    if any(part in {".venv", ".git", ".ipynb_checkpoints"} for part in path.parts):
        continue

    if path.suffix.lower() not in {".csv", ".json"}:
        continue

    name = path.name.lower()
    if any(term in name for term in summary_terms):
        summary_rows.append({
            "file": str(path.relative_to(ROOT)),
            "size_kb": round(path.stat().st_size / 1024, 1),
        })

summaries_df = pd.DataFrame(summary_rows)
print("Summary files found:", len(summaries_df))
display(summaries_df)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

EXCLUDE = {".venv", ".git", ".ipynb_checkpoints"}

def excluded(path):
    return any(part in EXCLUDE for part in path.parts)

summary_terms = (
    "summary", "comparison", "metrics", "evaluation",
    "results", "history"
)

records = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or excluded(path):
        continue
    if path.suffix.lower() not in {".csv", ".json"}:
        continue
    if not any(term in path.name.lower() for term in summary_terms):
        continue

    record = {
        "relative_path": str(path.relative_to(ROOT)),
        "format": path.suffix.lower(),
        "size_kb": round(path.stat().st_size / 1024, 1),
        "rows_or_items": None,
        "columns_or_keys": None,
        "preview": None,
        "read_status": "OK",
    }

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            record["rows_or_items"] = len(df)
            record["columns_or_keys"] = ", ".join(map(str, df.columns))
            record["preview"] = df.head(2).to_dict("records")
        else:
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)

            if isinstance(obj, list):
                record["rows_or_items"] = len(obj)
                record["columns_or_keys"] = (
                    ", ".join(map(str, obj[0].keys()))
                    if obj and isinstance(obj[0], dict)
                    else type(obj[0]).__name__ if obj else "empty list"
                )
                record["preview"] = obj[:1]
            elif isinstance(obj, dict):
                record["rows_or_items"] = len(obj)
                record["columns_or_keys"] = ", ".join(map(str, obj.keys()))
                record["preview"] = dict(list(obj.items())[:8])
            else:
                record["preview"] = str(obj)[:300]

    except Exception as exc:
        record["read_status"] = f"READ ERROR: {exc}"

    records.append(record)

audit_df = pd.DataFrame(records)

print("Total candidate result files:", len(audit_df))
print("\n=== COMPLETE FILE INVENTORY ===")
display(
    audit_df[
        ["relative_path", "format", "size_kb",
         "rows_or_items", "columns_or_keys", "read_status"]
    ]
)

# Save nothing: show the contents in manageable groups.
print("\n=== MODEL-RELATED CSV PREVIEWS ===")
model_mask = audit_df["relative_path"].str.contains(
    r"classical_ml|deep_learning|research_audit|validation|final_artifacts",
    case=False,
    regex=True,
)

for _, row in audit_df[model_mask].iterrows():
    print("\nFILE:", row["relative_path"])
    print("Preview:", str(row["preview"])[:1000])


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "Cycle-level models":
        "reports/research_audit/common_patient_split/overall_metrics.csv",
    "Patient-level models":
        "reports/research_audit/common_patient_split/patient_level_overall_metrics.csv",
}

for title, rel_path in files.items():
    path = ROOT / rel_path

    print("\n" + "=" * 90)
    print(title)

    if not path.exists():
        print("Not found:", rel_path)
        continue

    df = pd.read_csv(path)
    print("Columns:", df.columns.tolist())

    metric_cols = [
        c for c in [
            "model_id", "training_patients", "validation_patients_not_used",
            "test_patients", "test_recordings", "test_cycles",
            "accuracy", "balanced_accuracy", "macro_precision",
            "macro_recall", "macro_f1", "weighted_f1",
            "estimator_predict_ms_per_cycle", "deployed_artifact_bytes"
        ]
        if c in df.columns
    ]

    display(
        df[metric_cols].sort_values(
            "macro_f1", ascending=False
        ).reset_index(drop=True)
        if "macro_f1" in df.columns
        else df
    )

print("\n=== SAVED VALIDATION CHECKS ===")
validation_path = ROOT / "reports/validation/model_validation_summary.csv"

if validation_path.exists():
    display(pd.read_csv(validation_path))


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

print("=== COMMON-SPLIT AUDIT FILES ===")

audit_dir = ROOT / "reports/research_audit/common_patient_split"

for path in sorted(audit_dir.iterdir()):
    if path.is_file():
        print(path.name, "|", round(path.stat().st_size / 1024, 1), "KB")

print("\n=== MODEL CONFIGURATION / FEATURE METADATA ===")

for base in [
    ROOT / "reports/research_audit",
    ROOT / "models",
    ROOT / "data/processed/classical_ml",
]:
    if not base.exists():
        continue

    for path in sorted(base.rglob("*")):
        if not path.is_file():
            continue
        if any(part in {".venv", ".git", ".ipynb_checkpoints"} for part in path.parts):
            continue
        if path.suffix.lower() not in {".json", ".csv"}:
            continue

        name = path.name.lower()
        if any(term in name for term in [
            "config", "feature", "manifest", "split", "validation"
        ]):
            print("\nFILE:", path.relative_to(ROOT))

            try:
                if path.suffix.lower() == ".json":
                    with path.open("r", encoding="utf-8") as f:
                        obj = json.load(f)
                    print(json.dumps(obj, indent=2, default=str)[:3500])
                else:
                    df = pd.read_csv(path)
                    print("Shape:", df.shape)
                    print("Columns:", df.columns.tolist())
                    display(df.head(5))
            except Exception as exc:
                print("Could not read:", exc)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"

# 1. Read the common-split evaluation protocol
protocol_path = AUDIT / "protocol.json"

print("=" * 70)
print("COMMON-PATIENT-SPLIT PROTOCOL")
print("=" * 70)

if protocol_path.exists():
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    print(json.dumps(protocol, indent=2, default=str))
else:
    print("protocol.json not found")

# 2. Inspect per-class performance
per_class_path = AUDIT / "per_class_metrics.csv"

print("\n" + "=" * 70)
print("CYCLE-LEVEL PER-CLASS METRICS")
print("=" * 70)

if per_class_path.exists():
    per_class = pd.read_csv(per_class_path)
    print("Shape:", per_class.shape)
    print("Columns:", per_class.columns.tolist())
    display(per_class)
else:
    print("per_class_metrics.csv not found")

# 3. Inspect saved model artifact sizes
sizes_path = AUDIT / "model_artifact_sizes.csv"

print("\n" + "=" * 70)
print("SAVED MODEL ARTIFACT SIZES")
print("=" * 70)

if sizes_path.exists():
    sizes = pd.read_csv(sizes_path)
    print("Columns:", sizes.columns.tolist())
    display(sizes)
else:
    print("model_artifact_sizes.csv not found")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

SEARCH_DIRS = [
    ROOT / "models",
    ROOT / "reports",
    ROOT / "data/processed",
]

EXCLUDE_PARTS = {
    ".git", ".venv", "__pycache__", ".ipynb_checkpoints",
    "sound_phenotype_cnn",
}

interesting_suffixes = {
    ".pt", ".pth", ".joblib", ".pkl", ".cbm",
    ".json", ".csv", ".xlsx",
}

rows = []

for base in SEARCH_DIRS:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue
        if any(part in EXCLUDE_PARTS for part in path.parts):
            continue
        if path.suffix.lower() not in interesting_suffixes:
            continue

        name = path.name.lower()
        parent = str(path.parent.relative_to(ROOT)).lower()

        # Keep likely disease-classification artifacts and reports.
        keywords = (
            "disease", "classification", "xgboost", "experiment",
            "test_metrics", "test_summary", "comparison",
            "predictions", "checkpoint", "best_model",
            "patientgrouped", "patient_disjoint", "validation_metrics",
        )

        if any(word in name or word in parent for word in keywords):
            rows.append({
                "folder": str(path.parent.relative_to(ROOT)),
                "file": path.name,
                "extension": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 1),
            })

inventory = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .sort_values(["folder", "file"])
    .reset_index(drop=True)
)

print("Candidate saved artifacts and reports:", len(inventory))
display(inventory.to_string(index=False) if not inventory.empty else inventory)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

FILES = [
    "reports/experiment_manifest.json",
    "reports/final_artifacts/comparison_manifest.json",
    "reports/final_artifacts/cycle_level_model_comparison.csv",
    "reports/final_artifacts/patient_level_model_comparison.csv",
    "data/processed/deep_learning/final_deep_learning_comparison.csv",
    "data/processed/deep_learning/final_model_comparison_summary.json",
    "data/processed/classical_ml/final_classical_ml_comparison.csv",
    "data/processed/deep_learning/lightweight_multifeature_cnn/test_metrics.json",
]

for rel in FILES:
    path = ROOT / rel
    print("\n" + "=" * 85)
    print(rel)
    print("=" * 85)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        if path.suffix.lower() == ".json":
            obj = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(obj, indent=2, default=str)[:7000])
        elif path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df)
    except Exception as exc:
        print("Read error:", repr(exc))


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

candidates = {
    "Common audit predictions":
        "reports/research_audit/common_patient_split/test_predictions.csv",
    "Lightweight Mel CNN":
        "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    "CNN-LSTM":
        "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "Multi-feature CNN":
        "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Lightweight Multi-feature CNN":
        "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "Experiment A":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B":
        "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
}

for name, rel in candidates.items():
    path = ROOT / rel
    print("\n" + "=" * 75)
    print(name, "|", rel)
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    # Show identifier-like columns and a few rows to determine alignment.
    id_cols = [
        c for c in df.columns
        if any(k in c.lower() for k in
               ["patient", "recording", "audio_path", "filename", "file_path"])
    ]
    label_cols = [
        c for c in df.columns
        if any(k in c.lower() for k in
               ["true_label", "true_disease", "disease_label", "actual"])
    ]
    print("Potential ID columns:", id_cols)
    print("Potential true-label columns:", label_cols)
    display(df[id_cols + label_cols].head(3) if id_cols or label_cols else df.head(3))

# Locate disease-specific XGBoost artifacts and notebooks.
print("\n" + "=" * 75)
print("XGBOOST / PATIENT-GROUPED FILE DISCOVERY")
print("=" * 75)

for base in [ROOT / "models", ROOT / "reports", ROOT / "Notebooks 2"]:
    if not base.exists():
        continue
    for path in sorted(base.rglob("*")):
        if not path.is_file():
            continue
        if any(p in {".git", ".venv", "__pycache__", ".ipynb_checkpoints"}
               for p in path.parts):
            continue
        text = str(path.relative_to(ROOT)).lower()
        if "xgboost" in text or "patientgrouped" in text:
            print(path.relative_to(ROOT))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "audit": "reports/research_audit/common_patient_split/test_predictions.csv",
    "mel_cnn": "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    "cnn_lstm": "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "multifeature_cnn": "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "lightweight_multifeature": "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "experiment_A": "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "experiment_B": "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
}

dfs = {}
for name, rel in files.items():
    path = ROOT / rel
    if path.exists():
        dfs[name] = pd.read_csv(path)

# Use the saved Mel CNN manifest as the 1,561-row reference.
reference = dfs["mel_cnn"].copy().reset_index(drop=True)
ref_ids = ["patient_id", "recording_id", "chest_location"]
ref_labels = "true_label"

print("REFERENCE")
print("Rows:", len(reference))
print("Unique patient/recording/location keys:",
      reference[ref_ids].drop_duplicates().shape[0])

def compare_rows(name, df):
    df = df.reset_index(drop=True)
    print(f"\n{name}: {len(df)} rows")

    if len(df) != len(reference):
        print("Same row count:", False)
        return

    # Compare sample ordering and labels where matching columns exist.
    if all(c in df.columns for c in ref_ids):
        ids_equal = reference[ref_ids].astype(str).equals(
            df[ref_ids].astype(str)
        )
        print("Exact patient/recording/location row order:", ids_equal)
    elif "audio_path" in df.columns and "audio_path" in reference.columns:
        paths_equal = reference["audio_path"].astype(str).equals(
            df["audio_path"].astype(str)
        )
        print("Exact audio_path row order:", paths_equal)
    else:
        print("Full sample identifiers unavailable in this file.")

    label_col = next(
        (c for c in ["true_label", "disease_label"] if c in df.columns),
        None
    )
    if label_col:
        labels_equal = reference[ref_labels].astype(str).equals(
            df[label_col].astype(str)
        )
        print("Exact true-label sequence:", labels_equal)

# Compare standalone saved predictions.
for name in [
    "cnn_lstm", "multifeature_cnn",
    "lightweight_multifeature", "experiment_A", "experiment_B"
]:
    compare_rows(name, dfs[name])

# Compare each model's rows inside the common audit.
audit = dfs["audit"]
if "model_id" in audit.columns:
    print("\nCOMMON AUDIT MODEL ROWS")
    for model_id in sorted(audit["model_id"].dropna().unique()):
        sub = audit[audit["model_id"] == model_id].reset_index(drop=True)
        compare_rows(f"audit:{model_id}", sub)


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"

CLASS_NAMES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]

def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        if value in CLASS_NAMES:
            return value
        try:
            value = int(float(value))
        except ValueError:
            return value
    if isinstance(value, (int, float)) and int(value) in range(8):
        return CLASS_NAMES[int(value)]
    return str(value)

def calculate_metrics(name, y_true, y_pred):
    y_true = [normalize_label(x) for x in y_true]
    y_pred = [normalize_label(x) for x in y_pred]

    supported = [c for c in CLASS_NAMES if c in set(y_true)]
    return {
        "model": name,
        "n_test_cycles": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_supported": balanced_accuracy_score(y_true, y_pred),
        "macro_precision_supported": precision_score(
            y_true, y_pred, labels=supported,
            average="macro", zero_division=0
        ),
        "macro_recall_supported": recall_score(
            y_true, y_pred, labels=supported,
            average="macro", zero_division=0
        ),
        "macro_f1_supported": f1_score(
            y_true, y_pred, labels=supported,
            average="macro", zero_division=0
        ),
        "macro_f1_all8": f1_score(
            y_true, y_pred, labels=CLASS_NAMES,
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, labels=CLASS_NAMES,
            average="weighted", zero_division=0
        ),
        "classes_supported": len(supported),
    }

results = []

# Seven models from the verified common audit.
audit_path = AUDIT / "test_predictions.csv"
audit_df = pd.read_csv(audit_path)

for model_id, group in audit_df.groupby("model_id", sort=True):
    results.append(calculate_metrics(
        model_id,
        group["true_disease"],
        group["predicted_disease"],
    ))

# Additional models with saved predictions aligned to the same test sequence.
extra_files = {
    "Experiment A — weighted CE":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B — balanced sampler":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
    "Lightweight Multi-feature CNN":
        ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
}

for name, path in extra_files.items():
    df = pd.read_csv(path)
    true_col = "true_disease" if "true_disease" in df.columns else "true_class"
    pred_col = "predicted_disease" if "predicted_disease" in df.columns else "predicted_class"

    results.append(calculate_metrics(name, df[true_col], df[pred_col]))

comparison = pd.DataFrame(results).sort_values(
    ["macro_f1_supported", "balanced_accuracy_supported"],
    ascending=False
).reset_index(drop=True)

print("COMMON-TEST COMPARISON — sorted by supported-class macro F1")
display(comparison.round(4))

print("\nPREDICTED CLASS DISTRIBUTIONS")
for name, df in extra_files.items():
    frame = pd.read_csv(df)
    pred_col = "predicted_disease" if "predicted_disease" in frame.columns else "predicted_class"
    print(f"\n{name}")
    display(frame[pred_col].value_counts(dropna=False).to_frame("cycles_predicted"))


In [ ]:

from pathlib import Path
import json
import re

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
path = ROOT / "Notebooks 2/10_XGBoost_PatientGrouped_Classification.ipynb"

if not path.exists():
    print("Notebook not found:", path)
else:
    nb = json.loads(path.read_text(encoding="utf-8"))
    print("Notebook:", path.name)
    print("Total cells:", len(nb.get("cells", [])))

    keywords = re.compile(
        r"xgboost|xgbclassifier|train_test|patient|group|validation|"
        r"test_|predict|classification_report|macro.?f1|balanced_accuracy|"
        r"save|joblib|checkpoint|confusion_matrix",
        re.IGNORECASE,
    )

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        if keywords.search(source):
            print(f"\n{'=' * 24} CELL {i} ({cell.get('cell_type')}) {'=' * 24}")
            print(source[:3500])

            outputs = cell.get("outputs", [])
            if outputs:
                print("\n--- SAVED OUTPUT PREVIEW ---")
                for out in outputs[:2]:
                    text = out.get("text", [])
                    if text:
                        print("".join(text)[:1200])
                    data = out.get("data", {})
                    if "text/plain" in data:
                        value = data["text/plain"]
                        print("".join(value)[:1200] if isinstance(value, list) else str(value)[:1200])


In [ ]:

from pathlib import Path
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
path = ROOT / "Notebooks 2/10_XGBoost_PatientGrouped_Classification.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    print(f"\nCELL {i} | {cell.get('cell_type')} | {len(source)} characters")
    print(source[-1800:] if len(source) > 1800 else source)

    outputs = cell.get("outputs", [])
    for out in outputs:
        text = out.get("text", [])
        if text:
            print("SAVED OUTPUT:", "".join(text)[-1200:])


In [ ]:

from pathlib import Path

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

search_dirs = [
    ROOT / "models",
    ROOT / "data" / "processed",
    ROOT / "reports",
]

extensions = {
    ".json", ".joblib", ".pkl", ".pickle",
    ".ubj", ".model", ".csv", ".txt"
}

keywords = ("xgb", "xgboost", "patientgrouped")

print("=== XGBOOST-RELATED SAVED FILES ===")

found = []

for base in search_dirs:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        relative = str(path.relative_to(ROOT)).lower()

        if (
            path.suffix.lower() in extensions
            and any(key in name or key in relative for key in keywords)
        ):
            found.append(path)

if found:
    for path in sorted(found):
        print(
            f"{path.relative_to(ROOT)}"
            f" | {path.stat().st_size:,} bytes"
        )
else:
    print("No matching saved files found in the searched directories.")

print("\nTotal matching files:", len(found))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

features_path = (
    ROOT / "data/processed/features/respiratory_cycle_features.csv"
)
train_path = (
    ROOT / "data/processed/deep_learning/train_manifest.csv"
)
test_path = (
    ROOT / "data/processed/deep_learning/test_manifest.csv"
)

for path in [features_path, train_path, test_path]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

if all(p.exists() for p in [features_path, train_path, test_path]):
    features = pd.read_csv(features_path)
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    keys = [
        "patient_id",
        "recording_id",
        "chest_location",
        "diagnosis",
    ]

    train_patients = set(train["patient_id"].astype(str))
    test_patients = set(test["patient_id"].astype(str))

    print("\nFeature table rows:", len(features))
    print("Official train manifest rows:", len(train))
    print("Official test manifest rows:", len(test))
    print("Official test patients:", len(test_patients))
    print("Train/test patient overlap:", len(train_patients & test_patients))

    print("\nFeature columns available:", len(features.columns))
    print("Required join columns available:",
          all(c in features.columns and c in test.columns for c in keys))

    if all(c in features.columns and c in test.columns for c in keys):
        test_keys = test[keys].drop_duplicates()
        test_features = features.merge(
            test_keys,
            on=keys,
            how="inner",
            validate="many_to_one",
        )

        print("Matched test feature rows:", len(test_features))
        print("Test diagnoses:")
        print(test["diagnosis"].value_counts().sort_index())

        print("\nTest patient IDs overlap with development patients:",
              len(test_patients & train_patients))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

train = pd.read_csv(
    ROOT / "data/processed/deep_learning/train_manifest.csv"
)
test = pd.read_csv(
    ROOT / "data/processed/deep_learning/test_manifest.csv"
)

for df in (train, test):
    df["patient_id"] = df["patient_id"].astype(str)

overlap = sorted(
    set(train["patient_id"]) & set(test["patient_id"])
)

print("Overlapping patient IDs:", overlap)

for patient in overlap:
    print(f"\n{'=' * 55}")
    print("Patient:", patient)

    for split_name, df in [("TRAIN", train), ("TEST", test)]:
        rows = df[df["patient_id"] == patient]

        if rows.empty:
            continue

        print(f"\n{split_name}: {len(rows)} manifest rows")
        columns = [
            c for c in [
                "patient_id", "recording_id", "chest_location",
                "diagnosis", "official_split", "audio_path", "wav_path"
            ]
            if c in rows.columns
        ]

        display(rows[columns].drop_duplicates().head(20))

print("\nTrain manifest columns:", train.columns.tolist())
print("Test manifest columns:", test.columns.tolist())


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

train = pd.read_csv(
    ROOT / "data/processed/deep_learning/train_manifest.csv"
).fillna("")

test = pd.read_csv(
    ROOT / "data/processed/deep_learning/test_manifest.csv"
).fillna("")

for df in (train, test):
    df["patient_id"] = df["patient_id"].astype(str)
    for col in ["recording_id", "chest_location", "diagnosis", "audio_path"]:
        df[col] = df[col].astype(str).str.strip()

recording_keys = [
    "patient_id", "recording_id", "chest_location", "diagnosis"
]

train_recordings = set(map(tuple, train[recording_keys].drop_duplicates().values))
test_recordings = set(map(tuple, test[recording_keys].drop_duplicates().values))

train_audio = set(train["audio_path"]) - {""}
test_audio = set(test["audio_path"]) - {""}

print("Exact recording/location groups in both splits:",
      len(train_recordings & test_recordings))
print("Exact audio paths in both splits:",
      len(train_audio & test_audio))

print("\nShared recording/location groups:")
for row in sorted(train_recordings & test_recordings):
    print(row)

print("\nShared audio paths:")
for path in sorted(train_audio & test_audio):
    print(path)

print("\nPatients appearing in both manifests:")
print(sorted(set(train["patient_id"]) & set(test["patient_id"])))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

audit_dir = ROOT / "reports/research_audit/common_patient_split"

print("Audit directory exists:", audit_dir.exists())

if audit_dir.exists():
    for path in sorted(audit_dir.rglob("*")):
        if path.is_file():
            print(f"{path.relative_to(ROOT)} | {path.stat().st_size:,} bytes")

    metrics_path = audit_dir / "overall_metrics.csv"

    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path)
        print("\nExisting model IDs:")
        print(metrics["model_id"].tolist())

        print("\nExisting metric columns:")
        print(metrics.columns.tolist())


In [ ]:

from pathlib import Path
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
protocol_path = (
    ROOT / "reports/research_audit/common_patient_split/protocol.json"
)

protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

print(json.dumps(protocol, indent=2))

print("\n=== RELATED MANIFESTS ===")
for path in sorted((ROOT / "data/processed").rglob("*manifest*.csv")):
    print(path.relative_to(ROOT))


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

paths_to_check = [
    ROOT / "data/processed/features/respiratory_cycle_features.csv",
    ROOT / "data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv",
    ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv",
    ROOT / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv",
    ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv",
    ROOT / "Notebooks 2/10_XGBoost_PatientGrouped_Classification.ipynb",
]

for path in paths_to_check:
    print(f"\n{path.relative_to(ROOT)}")
    print("Exists:", path.exists())

# Inspect the audit's feature-selection record.
selection_path = (
    ROOT / "reports/research_audit/common_patient_split/"
    "classical_selected_features.json"
)

if selection_path.exists():
    print("\n=== CLASSICAL FEATURE SELECTION ===")
    print(json.dumps(
        json.loads(selection_path.read_text(encoding="utf-8")),
        indent=2
    ))

# Search only disease-classification locations for saved XGBoost artifacts.
print("\n=== POSSIBLE DISEASE XGBOOST ARTIFACTS ===")
search_roots = [
    ROOT / "models",
    ROOT / "data/processed",
    ROOT / "reports",
]

for base in search_roots:
    if base.exists():
        for path in base.rglob("*"):
            if path.is_file() and "xgboost" in str(path).lower():
                if "phenotype" not in str(path).lower():
                    print(path.relative_to(ROOT))


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
BASE = ROOT / "data/processed/deep_learning/cnn_baseline"
AUDIT = ROOT / "reports/research_audit/common_patient_split"

protocol = json.loads(
    (AUDIT / "protocol.json").read_text(encoding="utf-8")
)

files = {
    "train": BASE / "train_cnn_manifest.csv",
    "validation": BASE / "validation_cnn_manifest.csv",
    "test": BASE / "test_cnn_manifest.csv",
}

expected = {
    "train": set(map(str, protocol["train_patients"])),
    "validation": set(map(str, protocol["validation_patients_not_used_for_classical_fit"])),
    "test": set(map(str, protocol["test_patients"])),
}

patient_sets = {}

for split, path in files.items():
    df = pd.read_csv(path)
    patient_col = next(
        (c for c in ["patient_id", "patient", "patient_ID"]
         if c in df.columns),
        None
    )

    print(f"\n=== {split.upper()} ===")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if patient_col is None:
        print("No recognized patient ID column; cannot verify this manifest.")
        continue

    ids = set(df[patient_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True))
    patient_sets[split] = ids

    print("Unique patients:", len(ids))
    print("Matches audit patient set:", ids == expected[split])
    print("Missing from manifest:", sorted(expected[split] - ids))
    print("Extra in manifest:", sorted(ids - expected[split]))

print("\n=== PATIENT OVERLAP ===")
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    if a in patient_sets and b in patient_sets:
        print(f"{a} ∩ {b}:", sorted(patient_sets[a] & patient_sets[b]))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

features_path = ROOT / "data/processed/features/respiratory_cycle_features.csv"
train_path = ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv"
val_path = ROOT / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv"
test_path = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"

features = pd.read_csv(features_path, nrows=5)
train = pd.read_csv(train_path, nrows=5)

print("=== FEATURE TABLE ===")
print("Columns:", features.columns.tolist())
print(features.head(2).to_string(index=False))

print("\n=== MANIFEST ===")
print("Columns:", train.columns.tolist())
print(train.head(2).to_string(index=False))

# Read full files for row counts and key compatibility.
features = pd.read_csv(features_path)
manifests = {
    "train": pd.read_csv(train_path),
    "validation": pd.read_csv(val_path),
    "test": pd.read_csv(test_path),
}

print("\n=== ROW COUNTS ===")
print("Feature rows:", len(features))
for name, df in manifests.items():
    print(f"{name.capitalize()} manifest rows:", len(df))

print("\n=== POSSIBLE JOIN KEYS ===")
for candidate in ["audio_path", "recording_id", "patient_id"]:
    print(
        candidate,
        "| in features:", candidate in features.columns,
        "| in manifest:", candidate in manifests["train"].columns
    )

print("\n=== LABEL DISTRIBUTIONS ===")
for name, df in manifests.items():
    label_col = "disease_label" if "disease_label" in df.columns else "diagnosis"
    print(f"\n{name}:")
    print(df[label_col].value_counts(dropna=False).sort_index().to_string())


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

features = pd.read_csv(
    ROOT / "data/processed/features/respiratory_cycle_features.csv"
)

manifest_dir = ROOT / "data/processed/deep_learning/cnn_baseline"
train = pd.read_csv(manifest_dir / "train_cnn_manifest.csv")
val = pd.read_csv(manifest_dir / "validation_cnn_manifest.csv")
test = pd.read_csv(manifest_dir / "test_cnn_manifest.csv")

audit_dir = ROOT / "reports/research_audit/common_patient_split"
feature_list = json.loads(
    (audit_dir / "classical_selected_features.json").read_text()
)

keys = ["patient_id", "recording_id", "chest_location"]

print("=== FEATURE KEY UNIQUENESS ===")
print("Feature rows:", len(features))
print("Unique key combinations:", len(features[keys].drop_duplicates()))
print("Duplicate feature keys:", features.duplicated(keys).sum())

print("\n=== FEATURE AVAILABILITY ===")
missing_features = [c for c in feature_list if c not in features.columns]
print("Selected features:", len(feature_list))
print("Missing selected features:", missing_features)

print("\n=== MERGE CHECK ===")
for name, manifest in [
    ("train", train),
    ("validation", val),
    ("test", test),
]:
    merged = manifest.merge(
        features,
        on=keys,
        how="left",
        suffixes=("_manifest", "_feature"),
        validate="many_to_one",
        indicator=True,
    )
    print(
        f"{name}: manifest={len(manifest)}, merged={len(merged)}, "
        f"unmatched={int((merged['_merge'] != 'both').sum())}"
    )

    if "diagnosis_manifest" in merged.columns:
        mismatches = (
            merged["diagnosis_manifest"].astype(str)
            != merged["diagnosis_feature"].astype(str)
        )
        print("  Diagnosis mismatches:", int(mismatches.sum()))

print("\n=== LABEL MAPPING ===")
print(
    train[["disease_label", "diagnosis"]]
    .drop_duplicates()
    .sort_values("disease_label")
    .to_string(index=False)
)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

features = pd.read_csv(
    ROOT / "data/processed/features/respiratory_cycle_features.csv"
)

base = ROOT / "data/processed/deep_learning/cnn_baseline"

manifests = {
    "train": pd.read_csv(base / "train_cnn_manifest.csv"),
    "validation": pd.read_csv(base / "validation_cnn_manifest.csv"),
    "test": pd.read_csv(base / "test_cnn_manifest.csv"),
}

print("=== DUPLICATE FEATURE GROUPS ===")
keys = ["patient_id", "recording_id", "chest_location"]

group_sizes = features.groupby(keys, dropna=False).size()
print("Groups:", len(group_sizes))
print("Groups with multiple rows:", int((group_sizes > 1).sum()))
print("Largest group size:", int(group_sizes.max()))

print("\n=== FEATURE CYCLE NUMBER ===")
print("cycle_number column exists:", "cycle_number" in features.columns)
if "cycle_number" in features.columns:
    print("Missing cycle numbers:", int(features["cycle_number"].isna().sum()))
    print("Duplicate full keys including cycle_number:",
          int(features.duplicated(keys + ["cycle_number"]).sum()))
    print("\nExample feature rows:")
    print(
        features.loc[
            (features["patient_id"].astype(str) == "101")
            & (features["recording_id"].astype(str) == "1b1")
            & (features["chest_location"].astype(str) == "Al"),
            keys + ["cycle_number", "diagnosis", "processed_path"]
        ].head(12).to_string(index=False)
    )

print("\n=== MANIFEST CYCLE INFORMATION ===")
for name, df in manifests.items():
    print(f"\n{name}:")
    print("Has cycle_number:", "cycle_number" in df.columns)
    print("Has processed_path:", "processed_path" in df.columns)
    print("Has audio_path:", "audio_path" in df.columns)
    print(df.head(2).to_string(index=False))

print("\n=== RECORDING-LOCATION GROUP COUNTS ===")
for name, df in manifests.items():
    counts = df.groupby(keys, dropna=False).size()
    print(
        f"{name}: rows={len(df)}, groups={len(counts)}, "
        f"groups with multiple cycles={int((counts > 1).sum())}"
    )


In [ ]:

from pathlib import Path
import pandas as pd
import re

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

features = pd.read_csv(
    ROOT / "data/processed/features/respiratory_cycle_features.csv"
)

base = ROOT / "data/processed/deep_learning/cnn_baseline"
splits = {
    "train": pd.read_csv(base / "train_cnn_manifest.csv"),
    "validation": pd.read_csv(base / "validation_cnn_manifest.csv"),
    "test": pd.read_csv(base / "test_cnn_manifest.csv"),
}

# Derive the cycle number from the manifest filename, e.g. *_001.wav -> 1.
for name, df in splits.items():
    df["cycle_number"] = pd.to_numeric(
        df["audio_path"].str.extract(r"_(\d+)\.wav$", expand=False),
        errors="coerce"
    )

    print(
        f"{name}: missing parsed cycle numbers = "
        f"{int(df['cycle_number'].isna().sum())}"
    )

keys = ["patient_id", "recording_id", "chest_location", "cycle_number"]

# Normalize key types so integer IDs and cycle numbers match.
for df in [features, *splits.values()]:
    df["patient_id"] = df["patient_id"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["recording_id"] = df["recording_id"].astype(str)
    df["chest_location"] = df["chest_location"].astype(str)
    df["cycle_number"] = pd.to_numeric(df["cycle_number"], errors="coerce").astype("Int64")

print("\nFeature keys unique:", not features.duplicated(keys).any())

all_manifest = pd.concat(
    [df.assign(_split=name) for name, df in splits.items()],
    ignore_index=True
)

print("Total manifest rows:", len(all_manifest))
print("Total feature rows:", len(features))
print("Manifest duplicate keys:", int(all_manifest.duplicated(keys).sum()))

joined = all_manifest.merge(
    features[keys + ["diagnosis"] + [
        c for c in features.columns
        if c not in keys + ["diagnosis"]
    ]],
    on=keys,
    how="left",
    suffixes=("_manifest", "_feature"),
    validate="one_to_one",
    indicator=True
)

print("Joined rows:", len(joined))
print("Unmatched manifest rows:", int((joined["_merge"] != "both").sum()))

diagnosis_col = "diagnosis_feature"
if diagnosis_col in joined.columns:
    mismatch = (
        joined["diagnosis_manifest"].astype(str)
        != joined[diagnosis_col].astype(str)
    )
    print("Diagnosis mismatches:", int(mismatch.sum()))
else:
    print("Diagnosis columns:", [
        c for c in joined.columns if "diagnosis" in c.lower()
    ])

print("\nJoin status:")
print(joined["_merge"].value_counts().to_string())


In [ ]:

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier
import joblib

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
FEATURES = ROOT / "data/processed/features/respiratory_cycle_features.csv"
BASE = ROOT / "data/processed/deep_learning/cnn_baseline"
AUDIT = ROOT / "reports/research_audit/common_patient_split"
OUT = ROOT / "reports/research_audit/common_patient_split/xgboost"
OUT.mkdir(parents=True, exist_ok=True)

feature_cols = json.loads(
    (AUDIT / "classical_selected_features.json").read_text(encoding="utf-8")
)
features = pd.read_csv(FEATURES)

splits = {
    "train": pd.read_csv(BASE / "train_cnn_manifest.csv"),
    "validation": pd.read_csv(BASE / "validation_cnn_manifest.csv"),
    "test": pd.read_csv(BASE / "test_cnn_manifest.csv"),
}

# Derive cycle numbers from manifest filenames and normalize join keys.
for df in splits.values():
    df["cycle_number"] = pd.to_numeric(
        df["audio_path"].str.extract(r"_(\d+)\.wav$", expand=False),
        errors="raise"
    ).astype("int64")

for df in [features, *splits.values()]:
    df["patient_id"] = df["patient_id"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["recording_id"] = df["recording_id"].astype(str)
    df["chest_location"] = df["chest_location"].astype(str)
    df["cycle_number"] = pd.to_numeric(df["cycle_number"], errors="raise").astype("int64")

keys = ["patient_id", "recording_id", "chest_location", "cycle_number"]
assert not features.duplicated(keys).any()

feature_table = features[keys + feature_cols + ["diagnosis"]].copy()

for name, manifest in splits.items():
    merged = manifest.merge(
        feature_table,
        on=keys,
        how="left",
        suffixes=("_manifest", "_feature"),
        validate="one_to_one",
        indicator=True,
    )
    assert (merged["_merge"] == "both").all(), f"Unmatched {name} rows"
    assert (
        merged["diagnosis_manifest"].astype(str)
        == merged["diagnosis_feature"].astype(str)
    ).all(), f"Diagnosis mismatch in {name}"
    splits[name] = merged.drop(columns=["_merge"])

# Use numeric disease labels from the verified manifest.
label_names = (
    splits["train"][["disease_label", "diagnosis_manifest"]]
    .drop_duplicates()
    .sort_values("disease_label")
)
label_map = {
    int(row.disease_label): str(row.diagnosis_manifest)
    for row in label_names.itertuples(index=False)
}
print("Label mapping:", label_map)

X_train = splits["train"][feature_cols].astype(float)
y_train = splits["train"]["disease_label"].astype(int)
X_val = splits["validation"][feature_cols].astype(float)
y_val = splits["validation"]["disease_label"].astype(int)
X_test = splits["test"][feature_cols].astype(float)
y_test = splits["test"]["disease_label"].astype(int)

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_val.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()

# Selection uses validation macro-F1 only. No test metrics are used here.
configs = [
    {"max_depth": 2, "learning_rate": 0.05, "min_child_weight": 5, "reg_lambda": 5, "reg_alpha": 0.1},
    {"max_depth": 3, "learning_rate": 0.05, "min_child_weight": 5, "reg_lambda": 5, "reg_alpha": 0.1},
    {"max_depth": 4, "learning_rate": 0.05, "min_child_weight": 5, "reg_lambda": 5, "reg_alpha": 0.1},
    {"max_depth": 3, "learning_rate": 0.03, "min_child_weight": 5, "reg_lambda": 10, "reg_alpha": 0.5},
    {"max_depth": 3, "learning_rate": 0.08, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.5},
    {"max_depth": 2, "learning_rate": 0.08, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.5},
]

classes = np.array(sorted(y_train.unique()))
# Inverse-frequency weights, normalized to mean 1.
counts = y_train.value_counts()
class_weight = {int(c): len(y_train) / (len(counts) * int(counts[c])) for c in counts.index}
sample_weight = y_train.map(class_weight).to_numpy()

validation_results = []
models = []

for i, params in enumerate(configs, start=1):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(classes),
        n_estimators=350,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, sample_weight=sample_weight)
    pred_val = model.predict(X_val)

    # Macro-F1 over classes actually represented in validation.
    supported = sorted(y_val.unique())
    score = f1_score(y_val, pred_val, labels=supported, average="macro", zero_division=0)
    validation_results.append({
        "config_id": i,
        **params,
        "validation_accuracy": accuracy_score(y_val, pred_val),
        "validation_balanced_accuracy": balanced_accuracy_score(y_val, pred_val),
        "validation_macro_f1_supported": score,
        "validation_weighted_f1": f1_score(y_val, pred_val, average="weighted", zero_division=0),
    })
    models.append(model)
    print(f"Config {i}/{len(configs)}: validation supported macro-F1={score:.4f}")

results = pd.DataFrame(validation_results).sort_values(
    ["validation_macro_f1_supported", "validation_balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

best_id = int(results.loc[0, "config_id"])
best_model = models[best_id - 1]

print("\n=== VALIDATION CONFIGURATION RESULTS ===")
print(results.to_string(index=False))

# Final test evaluation: run once, after configuration selection.
pred_test = best_model.predict(X_test)
supported_test = sorted(y_test.unique())

test_metrics = {
    "selected_config_id": best_id,
    "n_test_cycles": int(len(y_test)),
    "test_classes_supported": [label_map[int(c)] for c in supported_test],
    "accuracy": float(accuracy_score(y_test, pred_test)),
    "balanced_accuracy_supported": float(
        balanced_accuracy_score(y_test, pred_test)
    ),
    "macro_f1_supported": float(
        f1_score(y_test, pred_test, labels=supported_test, average="macro", zero_division=0)
    ),
    "macro_f1_all8": float(
        f1_score(y_test, pred_test, labels=sorted(label_map), average="macro", zero_division=0)
    ),
    "weighted_f1": float(f1_score(y_test, pred_test, average="weighted", zero_division=0)),
}

print("\n=== FINAL TEST METRICS ===")
print(json.dumps(test_metrics, indent=2))
print("\n=== PER-CLASS TEST REPORT ===")
print(classification_report(
    y_test, pred_test,
    labels=sorted(label_map),
    target_names=[label_map[c] for c in sorted(label_map)],
    zero_division=0
))

# Save new XGBoost artifacts without modifying existing model outputs.
joblib.dump(best_model, OUT / "xgboost_disease_model.joblib")
results.to_csv(OUT / "validation_config_results.csv", index=False)

predictions = splits["test"][[
    "patient_id", "recording_id", "chest_location",
    "cycle_number", "diagnosis_manifest", "disease_label"
]].copy()
predictions = predictions.rename(columns={
    "diagnosis_manifest": "true_disease",
    "disease_label": "true_label",
})
predictions["predicted_label"] = pred_test
predictions["predicted_disease"] = [
    label_map[int(x)] for x in pred_test
]
predictions.to_csv(OUT / "xgboost_test_predictions.csv", index=False)

(OUT / "xgboost_test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2), encoding="utf-8"
)

print("\nSaved artifacts to:", OUT)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"
XGB_DIR = AUDIT / "xgboost"

reference = pd.read_csv(AUDIT / "test_predictions.csv")
xgb = pd.read_csv(XGB_DIR / "xgboost_test_predictions.csv")

print("=== REFERENCE PREDICTION FILE ===")
print("Rows:", len(reference))
print("Columns:", reference.columns.tolist())
print("Rows per model:")
print(reference["model_id"].value_counts().to_string())

print("\n=== REFERENCE KEY DUPLICATES ===")
keys = ["patient_id", "recording_id", "chest_location"]
ref_lr = reference[reference["model_id"] == "logistic_regression"].copy()

print("Logistic regression rows:", len(ref_lr))
print("Duplicate 3-part keys:", ref_lr.duplicated(keys).sum())

# Show whether the same keys repeat across different respiratory cycles.
print("\nExample repeated keys:")
print(
    ref_lr.loc[ref_lr.duplicated(keys, keep=False), keys + ["true_label"]]
    .head(12)
    .to_string(index=False)
)

print("\n=== XGBOOST PREDICTION FILE ===")
print("Rows:", len(xgb))
print("Columns:", xgb.columns.tolist())
print(xgb.head(3).to_string(index=False))

print("\n=== CYCLE IDENTIFIERS ===")
print("Reference has cycle_number:", "cycle_number" in ref_lr.columns)
print("XGBoost has cycle_number:", "cycle_number" in xgb.columns)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"
BASE = ROOT / "data/processed/deep_learning/cnn_baseline"

reference = pd.read_csv(AUDIT / "test_predictions.csv")
xgb = pd.read_csv(AUDIT / "xgboost/xgboost_test_predictions.csv")
test_manifest = pd.read_csv(BASE / "test_cnn_manifest.csv")

keys = ["patient_id", "recording_id", "chest_location", "cycle_number"]

# Reconstruct cycle number from canonical test audio paths.
test_manifest["cycle_number"] = pd.to_numeric(
    test_manifest["audio_path"].str.extract(r"_(\d+)\.wav$", expand=False),
    errors="raise"
).astype(int)

for df in [reference, xgb, test_manifest]:
    df["patient_id"] = df["patient_id"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["recording_id"] = df["recording_id"].astype(str)
    df["chest_location"] = df["chest_location"].astype(str)
    df["true_label"] = df["true_label"].astype(int) if "true_label" in df else df.get("disease_label", pd.Series(dtype=int))

test_manifest["disease_label"] = test_manifest["disease_label"].astype(int)
xgb["cycle_number"] = xgb["cycle_number"].astype(int)

canonical = test_manifest[keys + ["disease_label", "diagnosis"]].copy()
canonical = canonical.rename(columns={
    "disease_label": "canonical_label",
    "diagnosis": "canonical_disease",
})

print("Canonical test rows:", len(canonical))
print("Canonical keys unique:", not canonical.duplicated(keys).any())

print("\n=== XGBOOST ALIGNMENT ===")
xgb_check = canonical.merge(
    xgb[keys + ["true_label", "predicted_label"]],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)
print("All XGBoost keys matched:", bool((xgb_check["_merge"] == "both").all()))
print("XGBoost label mismatches:",
      int((xgb_check["canonical_label"] != xgb_check["true_label"]).sum()))

print("\n=== EXISTING MODEL ALIGNMENT ===")
for model_id, group in reference.groupby("model_id"):
    # The reference lacks cycle_number, so compare its ordered row labels
    # with the canonical test sequence. The audit protocol says the saved
    # neural predictions were row-aligned to the canonical test loader.
    group = group.reset_index(drop=True)
    same_length = len(group) == len(canonical)
    same_labels = same_length and (
        group["true_label"].to_numpy() == canonical["canonical_label"].to_numpy()
    )
    print(
        f"{model_id}: rows={len(group)}, "
        f"canonical row count matches={same_length}, "
        f"true-label sequence matches={same_labels}"
    )


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"
XGB_DIR = AUDIT / "xgboost"
OUT = AUDIT / "final_comparison"
OUT.mkdir(parents=True, exist_ok=True)

reference = pd.read_csv(AUDIT / "test_predictions.csv")
xgb = pd.read_csv(XGB_DIR / "xgboost_test_predictions.csv")
canonical = pd.read_csv(
    ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"
)

canonical["cycle_number"] = pd.to_numeric(
    canonical["audio_path"].str.extract(r"_(\d+)\.wav$", expand=False),
    errors="raise"
).astype(int)

for df in [reference, xgb, canonical]:
    for c in ["patient_id", "recording_id", "chest_location"]:
        df[c] = df[c].astype(str).str.replace(r"\.0$", "", regex=True)

keys = ["patient_id", "recording_id", "chest_location", "cycle_number"]
canonical["disease_label"] = canonical["disease_label"].astype(int)
xgb["cycle_number"] = xgb["cycle_number"].astype(int)
xgb["true_label"] = xgb["true_label"].astype(int)
xgb["predicted_label"] = xgb["predicted_label"].astype(int)

canonical = canonical.sort_values(keys).reset_index(drop=True)
xgb = xgb.sort_values(keys).reset_index(drop=True)

# Explicit key and true-label alignment checks.
assert canonical[keys].equals(xgb[keys]), "XGBoost cycle keys differ from canonical test."
assert np.array_equal(
    canonical["disease_label"].to_numpy(),
    xgb["true_label"].to_numpy()
), "XGBoost true labels differ from canonical test."

label_names = {
    0: "Asthma", 1: "Bronchiectasis", 2: "Bronchiolitis",
    3: "COPD", 4: "Healthy", 5: "LRTI",
    6: "Pneumonia", 7: "URTI"
}
all_labels = list(label_names)
true = xgb["true_label"].to_numpy()
pred = xgb["predicted_label"].to_numpy()
supported = sorted(np.unique(true).tolist())

# Existing audited metrics + the newly computed XGBoost row.
leaderboard = pd.read_csv(AUDIT / "overall_metrics.csv")
xgb_row = {
    "model_id": "xgboost",
    "test_cycles": len(true),
    "accuracy": accuracy_score(true, pred),
    "balanced_accuracy": balanced_accuracy_score(true, pred),
    "macro_precision": precision_score(
        true, pred, labels=supported, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        true, pred, labels=supported, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        true, pred, labels=supported, average="macro", zero_division=0
    ),
    "macro_f1_all8": f1_score(
        true, pred, labels=all_labels, average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(true, pred, average="weighted", zero_division=0),
}

for col in leaderboard.columns:
    if col not in xgb_row:
        xgb_row[col] = np.nan

leaderboard = pd.concat(
    [leaderboard[leaderboard["model_id"] != "xgboost"],
     pd.DataFrame([xgb_row])[leaderboard.columns]],
    ignore_index=True
)

leaderboard.to_csv(OUT / "overall_metrics_with_xgboost.csv", index=False)

# Per-class metrics and confusion matrix.
per_class = pd.DataFrame({
    "label_id": all_labels,
    "class": [label_names[i] for i in all_labels],
    "support": [(true == i).sum() for i in all_labels],
    "precision": precision_score(
        true, pred, labels=all_labels, average=None, zero_division=0
    ),
    "recall": recall_score(
        true, pred, labels=all_labels, average=None, zero_division=0
    ),
    "f1": f1_score(
        true, pred, labels=all_labels, average=None, zero_division=0
    ),
})
per_class.to_csv(OUT / "per_class_metrics_xgboost.csv", index=False)

cm = confusion_matrix(true, pred, labels=all_labels)
pd.DataFrame(
    cm,
    index=[f"true_{label_names[i]}" for i in all_labels],
    columns=[f"pred_{label_names[i]}" for i in all_labels],
).to_csv(OUT / "confusion_matrix_xgboost.csv")

print("Alignment verified: 1,561/1,561 test cycles.")
print("\n=== LEADERBOARD (sorted by macro-F1) ===")
print(
    leaderboard.sort_values("macro_f1", ascending=False)[
        ["model_id", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].to_string(index=False)
)
print("\n=== XGBOOST PER-CLASS METRICS ===")
print(per_class.to_string(index=False))
print("\nReports saved to:", OUT)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

search_roots = [
    ROOT / "models",
    ROOT / "data/processed/deep_learning",
    ROOT / "reports",
]

extensions = {
    ".pt", ".pth", ".keras", ".h5", ".joblib", ".pkl",
    ".csv", ".json", ".xlsx"
}

keywords = [
    "prediction", "metric", "summary", "comparison",
    "experiment_a", "experiment_b", "experiment_c1",
    "experiment_c2", "lightweight", "multifeature",
    "cnn", "xgboost", "result", "evaluation",
]

rows = []

for base in search_roots:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        # Exclude cache directories and environments.
        if any(part in {".venv", "__pycache__", ".git"} for part in path.parts):
            continue

        name = path.name.lower()
        relative = str(path.relative_to(ROOT))

        if any(k in name or k in relative.lower() for k in keywords):
            rows.append({
                "file": relative,
                "type": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 1),
            })

inventory = pd.DataFrame(rows).drop_duplicates("file").sort_values("file")

print("Total candidate artifacts:", len(inventory))
print("\n=== SAVED MODEL / RESULT ARTIFACT INVENTORY ===")
print(inventory.to_string(index=False))

out = ROOT / "reports/research_audit/all_experiment_artifact_inventory.csv"
out.parent.mkdir(parents=True, exist_ok=True)
inventory.to_csv(out, index=False)
print("\nInventory saved to:", out)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = [
    # Common patient-holdout experiments
    "reports/research_audit/common_patient_split/overall_metrics.csv",
    "reports/research_audit/common_patient_split/patient_level_overall_metrics.csv",
    "reports/research_audit/common_patient_split/final_comparison/overall_metrics_with_xgboost.csv",
    "data/processed/deep_learning/lightweight_multifeature_cnn/test_metrics.json",
    "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/overall_metrics.csv",

    # Deep-learning summaries
    "data/processed/deep_learning/final_deep_learning_comparison.csv",
    "data/processed/deep_learning/final_model_comparison_summary.json",
    "data/processed/deep_learning/multifeature_cnn/final_cnn_experiment_comparison.csv",
    "data/processed/deep_learning/multifeature_cnn/disease_level_cnn_comparison.csv",
    "data/processed/deep_learning/multifeature_cnn/cnn_baseline_vs_multifeature_comparison.csv",
    "data/processed/deep_learning/patient_level_prediction/final_patient_level_comparison.csv",
    "data/processed/deep_learning/patient_level_prediction/patient_aggregation_comparison.csv",

    # Combined ICBHI/Fraiwan experiments
    "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/experiment_b_test_summary.csv",
    "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/evaluation/c1_test_summary.csv",
    "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/evaluation_c1_normalized/summary.csv",
]

for rel in files:
    path = ROOT / rel
    print("\n" + "=" * 100)
    print(rel)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print(df.to_string(index=False, max_rows=40))
        elif path.suffix.lower() == ".json":
            obj = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(obj, indent=2)[:12000])
    except Exception as exc:
        print("Could not read:", repr(exc))

# C2 metrics may be stored in several evaluation files, so list them.
print("\n" + "=" * 100)
print("C2 AND OTHER COMBINED EXPERIMENT RESULT FILES")

combined = ROOT / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
for path in sorted(combined.rglob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["summary", "metric", "classification_report"]
    ):
        if path.suffix.lower() in {".csv", ".json"}:
            print(path.relative_to(ROOT))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = {
    "lightweight_multifeature_cnn":
        ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "experiment_A":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "experiment_B":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
    "multifeature_cnn":
        ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "regularized_multifeature_cnn":
        ROOT / "data/processed/deep_learning/multifeature_cnn_12b/test_predictions.csv",
}

for name, path in files.items():
    print("\n" + "=" * 85)
    print(name)
    print("Path exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

    for col in ["patient_id", "recording_id", "chest_location",
                "cycle_number", "audio_path", "true_label",
                "predicted_label", "true_disease", "predicted_disease"]:
        if col in df.columns:
            print(f"  {col}: present")


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

MANIFEST = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"

FILES = {
    "Lightweight Multi-feature CNN":
        ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "Experiment A — Weighted CE":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B — Balanced Sampler":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
    "Multi-feature CNN":
        ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Regularized Multi-feature CNN":
        ROOT / "data/processed/deep_learning/multifeature_cnn_12b/test_predictions.csv",
}

LABELS = list(range(8))
CLASS_NAMES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]

def find_column(df, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise ValueError(
            f"Cannot find any of {candidates}. Available columns: {df.columns.tolist()}"
        )
    return None

def cycle_key(path):
    """Normalize absolute/relative audio paths to a comparable cycle identifier."""
    if pd.isna(path):
        return None
    s = str(path).replace("\\", "/")
    marker = "audio/cycles/"
    if marker in s:
        s = s.split(marker, 1)[1]
    return s.lstrip("./")

if not MANIFEST.exists():
    raise FileNotFoundError(f"Canonical test manifest not found: {MANIFEST}")

manifest = pd.read_csv(MANIFEST)
manifest_path_col = find_column(
    manifest, ["audio_path", "processed_path", "wav_path"], required=False
)
manifest_y_col = find_column(
    manifest, ["true_label", "disease_label", "label", "diagnosis"]
)

if manifest_path_col:
    manifest_keys = manifest[manifest_path_col].map(cycle_key).tolist()
else:
    manifest_keys = None

manifest_y = manifest[manifest_y_col].copy()
if not pd.api.types.is_numeric_dtype(manifest_y):
    label_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
    manifest_y = manifest_y.map(
        lambda v: label_to_id.get(str(v), v)
    )
manifest_y = pd.to_numeric(manifest_y, errors="raise").astype(int).to_numpy()

print("Canonical manifest rows:", len(manifest))
print("Manifest label column:", manifest_y_col)
print("Manifest audio-path column:", manifest_path_col)
print("Canonical class support:")
print(pd.Series(manifest_y).map(dict(enumerate(CLASS_NAMES))).value_counts().sort_index())

results = []

for model_name, path in FILES.items():
    print("\n" + "=" * 80)
    print(model_name)

    if not path.exists():
        print("MISSING FILE:", path)
        continue

    df = pd.read_csv(path)
    y_col = find_column(df, ["true_label", "disease_label", "label"])
    p_col = find_column(df, ["predicted_label", "prediction", "pred_label"])

    y = pd.to_numeric(df[y_col], errors="raise").astype(int).to_numpy()
    pred = pd.to_numeric(df[p_col], errors="raise").astype(int).to_numpy()

    if len(y) != len(pred):
        print("ERROR: truth/prediction length mismatch")
        continue

    # Verify truth labels against canonical manifest.
    labels_match = len(y) == len(manifest_y) and np.array_equal(y, manifest_y)

    # Verify exact cycle alignment if the prediction file has audio paths.
    pred_path_col = find_column(
        df, ["audio_path", "processed_path", "wav_path"], required=False
    )
    path_alignment = "UNVERIFIED — no audio-path column in predictions"

    if pred_path_col and manifest_keys is not None:
        pred_keys = df[pred_path_col].map(cycle_key).tolist()
        path_alignment = (
            "MATCH" if pred_keys == manifest_keys else "MISMATCH"
        )

    print("Rows:", len(df))
    print("Truth labels match canonical manifest in order:", labels_match)
    print("Exact audio-cycle path alignment:", path_alignment)

    if not np.isin(y, LABELS).all() or not np.isin(pred, LABELS).all():
        print("ERROR: labels outside expected class IDs 0–7; skipping metrics")
        continue

    present_labels = sorted(np.unique(y).tolist())
    row = {
        "model": model_name,
        "n_test": len(y),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy_supported": balanced_accuracy_score(y, pred),
        "macro_precision_supported": precision_score(
            y, pred, labels=present_labels, average="macro", zero_division=0
        ),
        "macro_recall_supported": recall_score(
            y, pred, labels=present_labels, average="macro", zero_division=0
        ),
        "macro_f1_supported": f1_score(
            y, pred, labels=present_labels, average="macro", zero_division=0
        ),
        "macro_f1_all8": f1_score(
            y, pred, labels=LABELS, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y, pred, labels=LABELS, average="weighted", zero_division=0
        ),
        "classes_present": len(present_labels),
        "truth_labels_match_manifest": labels_match,
        "exact_path_alignment": path_alignment,
    }
    results.append(row)

if results:
    result_df = pd.DataFrame(results)
    display_cols = [
        "model", "n_test", "accuracy", "balanced_accuracy_supported",
        "macro_f1_supported", "macro_f1_all8", "weighted_f1",
        "truth_labels_match_manifest", "exact_path_alignment"
    ]
    print("\n\n========== RECOMPUTED METRICS ==========")
    print(result_df[display_cols].round(4).to_string(index=False))

    out_dir = ROOT / "reports/research_audit/common_patient_split/final_comparison"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_dir / "recomputed_additional_model_metrics.csv"
    result_df.to_csv(out_csv, index=False)
    print("\nSaved:", out_csv)
    print("\nMetric definitions:")
    print("- supported macro-F1: averages only classes present in the true test labels")
    print("- all-8 macro-F1: averages all eight disease classes; absent classes contribute zero")
    print("- balanced accuracy: average recall over classes present in the true labels")
else:
    print("No model metrics were generated. Check the file paths and columns.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"
OUT = AUDIT / "final_comparison"
OUT.mkdir(parents=True, exist_ok=True)

N_CLASSES = 8
ALL_LABELS = list(range(N_CLASSES))

# Canonical test manifest
manifest = pd.read_csv(
    ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"
)
true_col = next(
    c for c in ["disease_label", "true_label", "label", "diagnosis"]
    if c in manifest.columns
)
canonical_y = pd.to_numeric(manifest[true_col], errors="raise").astype(int).to_numpy()

def metric_row(name, y, pred, alignment):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)

    if len(y) != len(canonical_y) or len(pred) != len(canonical_y):
        raise ValueError(
            f"{name}: expected {len(canonical_y)} rows; got y={len(y)}, pred={len(pred)}"
        )

    labels_present = sorted(np.unique(y).tolist())
    return {
        "model": name,
        "test_cycles": len(y),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy_supported": balanced_accuracy_score(y, pred),
        "macro_precision_supported": precision_score(
            y, pred, labels=labels_present, average="macro", zero_division=0
        ),
        "macro_recall_supported": recall_score(
            y, pred, labels=labels_present, average="macro", zero_division=0
        ),
        "macro_f1_supported": f1_score(
            y, pred, labels=labels_present, average="macro", zero_division=0
        ),
        "macro_f1_all8": f1_score(
            y, pred, labels=ALL_LABELS, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y, pred, labels=ALL_LABELS, average="weighted", zero_division=0
        ),
        "truth_sequence_matches_manifest": np.array_equal(y, canonical_y),
        "alignment_evidence": alignment,
    }

rows = []
alignment_checks = []

# 1) Seven models from the common patient split audit
audit_pred_path = AUDIT / "test_predictions.csv"
audit_df = pd.read_csv(audit_pred_path)

for model_id, g in audit_df.groupby("model_id", sort=False):
    g = g.reset_index(drop=True)
    y = pd.to_numeric(g["true_label"], errors="raise").astype(int).to_numpy()
    pred = pd.to_numeric(g["predicted_label"], errors="raise").astype(int).to_numpy()

    # The audit file has patient/recording/location, but not cycle_number.
    # Truth sequence comparison checks row order consistency, not unique cycle identity.
    truth_match = np.array_equal(y, canonical_y)
    alignment = (
        "truth sequence matches; cycle-level identity not fully verifiable"
        if truth_match else "WARNING: truth sequence differs from canonical manifest"
    )

    rows.append(metric_row(model_id, y, pred, alignment))
    alignment_checks.append({
        "model": model_id,
        "rows": len(g),
        "truth_sequence_matches_manifest": truth_match,
        "alignment_evidence": alignment,
    })

# 2) Additional saved predictions
additional_files = {
    "Lightweight Multi-feature CNN":
        ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "Experiment A — Weighted CE":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_A_weighted_ce_test_predictions.csv",
    "Experiment B — Balanced Sampler":
        ROOT / "data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions/experiment_B_balanced_sampler_test_predictions.csv",
    "Multi-feature CNN":
        ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "Regularized Multi-feature CNN":
        ROOT / "data/processed/deep_learning/multifeature_cnn_12b/test_predictions.csv",
    "XGBoost":
        AUDIT / "xgboost/xgboost_test_predictions.csv",
}

for name, path in additional_files.items():
    if not path.exists():
        print(f"NOT FOUND — skipping {name}: {path}")
        continue

    df = pd.read_csv(path)
    y_col = next((c for c in ["true_label", "disease_label"] if c in df.columns), None)
    pred_col = next((c for c in ["predicted_label", "prediction"] if c in df.columns), None)

    if y_col is None or pred_col is None:
        print(f"UNRECOGNIZED PREDICTION COLUMNS — skipping {name}: {df.columns.tolist()}")
        continue

    y = pd.to_numeric(df[y_col], errors="raise").astype(int).to_numpy()
    pred = pd.to_numeric(df[pred_col], errors="raise").astype(int).to_numpy()

    truth_match = len(y) == len(canonical_y) and np.array_equal(y, canonical_y)

    path_col = next((c for c in ["audio_path", "processed_path", "wav_path"] if c in df.columns), None)
    if path_col:
        mpath_col = next((c for c in ["audio_path", "processed_path", "wav_path"] if c in manifest.columns), None)
        if mpath_col:
            def norm_path(v):
                s = str(v).replace("\\", "/")
                marker = "audio/cycles/"
                if marker in s:
                    s = s.split(marker, 1)[1]
                return s.lstrip("./")
            path_match = (
                df[path_col].map(norm_path).tolist()
                == manifest[mpath_col].map(norm_path).tolist()
            )
            alignment = "exact audio-path MATCH" if path_match else "WARNING: audio-path MISMATCH"
        else:
            alignment = "audio path present; manifest path unavailable"
    else:
        alignment = "no audio path in prediction file; identity not independently verified"

    rows.append(metric_row(name, y, pred, alignment))
    alignment_checks.append({
        "model": name,
        "rows": len(df),
        "truth_sequence_matches_manifest": truth_match,
        "alignment_evidence": alignment,
    })

leaderboard = pd.DataFrame(rows)

# Prevent silent duplicates if a model appears in more than one source.
if leaderboard["model"].duplicated().any():
    print("WARNING: duplicate model names found:")
    print(leaderboard.loc[leaderboard["model"].duplicated(False), "model"].to_list())

leaderboard = leaderboard.sort_values(
    ["macro_f1_supported", "balanced_accuracy_supported"],
    ascending=False
).reset_index(drop=True)

leaderboard.insert(0, "rank_by_supported_macro_f1", np.arange(1, len(leaderboard) + 1))

leaderboard_path = OUT / "complete_common_test_leaderboard.csv"
alignment_path = OUT / "complete_common_test_alignment_checks.csv"

leaderboard.to_csv(leaderboard_path, index=False)
pd.DataFrame(alignment_checks).to_csv(alignment_path, index=False)

cols = [
    "rank_by_supported_macro_f1", "model", "test_cycles",
    "accuracy", "balanced_accuracy_supported",
    "macro_f1_supported", "macro_f1_all8", "weighted_f1",
    "alignment_evidence"
]
print("\n========== COMPLETE COMMON-TEST LEADERBOARD ==========")
print(leaderboard[cols].round(4).to_string(index=False))

print("\n========== ALIGNMENT CHECKS ==========")
print(pd.DataFrame(alignment_checks).to_string(index=False))

print("\nSaved leaderboard:", leaderboard_path)
print("Saved alignment checks:", alignment_path)
print("\nReminder: the test set contains no Asthma or LRTI examples.")
print("Supported macro-F1 excludes those absent classes; all-8 macro-F1 includes them as zero.")
print("This is a retrospective comparison, not a pristine untouched benchmark if earlier test results informed model choices.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
AUDIT = ROOT / "reports/research_audit/common_patient_split"
OUT = AUDIT / "final_comparison"

leaderboard_path = OUT / "complete_common_test_leaderboard.csv"
manifest_path = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"
xgb_path = AUDIT / "xgboost/xgboost_test_predictions.csv"

leaderboard = pd.read_csv(leaderboard_path)
manifest = pd.read_csv(manifest_path)
xgb = pd.read_csv(xgb_path)

# Normalize alternate names for the same model.
canonical_names = {
    "Lightweight Multi-feature CNN": "lightweight_multifeature_cnn",
    "lightweight_multifeature_cnn": "lightweight_multifeature_cnn",
    "Multi-feature CNN": "multifeature_cnn",
    "multifeature_cnn": "multifeature_cnn",
    "Regularized Multi-feature CNN": "regularized_multifeature_cnn",
    "regularized_multifeature_cnn": "regularized_multifeature_cnn",
    "Experiment A — Weighted CE": "experiment_A_weighted_ce",
    "Experiment B — Balanced Sampler": "experiment_B_balanced_sampler",
    "XGBoost": "xgboost",
    "logistic_regression": "logistic_regression",
    "svm": "svm",
    "random_forest": "random_forest",
    "lightweight_mel_cnn": "lightweight_mel_cnn",
    "cnn_lstm": "cnn_lstm",
}

leaderboard["canonical_model_id"] = leaderboard["model"].map(
    lambda name: canonical_names.get(name, name)
)

# Remove duplicates; retain one entry per actual model.
dedup = leaderboard.drop_duplicates(
    subset="canonical_model_id", keep="first"
).copy()

# Recalculate rank safely, even if a rank column already exists.
dedup = dedup.drop(
    columns=["rank_by_supported_macro_f1"], errors="ignore"
)
dedup = dedup.sort_values(
    ["macro_f1_supported", "balanced_accuracy_supported"],
    ascending=False
).reset_index(drop=True)
dedup.insert(0, "rank_by_supported_macro_f1", range(1, len(dedup) + 1))

final_path = OUT / "deduplicated_common_test_leaderboard.csv"
dedup.to_csv(final_path, index=False)

print("========== DEDUPLICATED LEADERBOARD ==========")
cols = [
    "rank_by_supported_macro_f1", "canonical_model_id", "test_cycles",
    "accuracy", "balanced_accuracy_supported",
    "macro_f1_supported", "macro_f1_all8", "weighted_f1"
]
print(dedup[cols].round(4).to_string(index=False))
print("\nUnique models:", dedup["canonical_model_id"].nunique())
print("Saved:", final_path)

# Verify XGBoost keys against the canonical manifest.
key_cols = ["patient_id", "recording_id", "chest_location", "cycle_number"]

def cycle_number_from_path(path):
    # Example suffix: .../1b1_Al_001.wav -> cycle_number 1
    import re
    match = re.search(r"_(\d+)\.wav$", str(path), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

if "cycle_number" not in manifest.columns and "audio_path" in manifest.columns:
    manifest["cycle_number"] = manifest["audio_path"].map(cycle_number_from_path)

missing_manifest_keys = [c for c in key_cols if c not in manifest.columns]
missing_xgb_keys = [c for c in key_cols if c not in xgb.columns]

print("\n========== XGBOOST KEY ALIGNMENT ==========")
if missing_manifest_keys or missing_xgb_keys:
    print("Cannot complete key check.")
    print("Missing manifest columns:", missing_manifest_keys)
    print("Missing XGBoost columns:", missing_xgb_keys)
else:
    mk = manifest[key_cols].copy()
    xk = xgb[key_cols].copy()

    for df in (mk, xk):
        for col in ["patient_id", "recording_id", "chest_location"]:
            df[col] = df[col].astype(str).str.strip()
        df["cycle_number"] = pd.to_numeric(
            df["cycle_number"], errors="coerce"
        ).astype("Int64")

    print("Manifest duplicate keys:", int(mk.duplicated().sum()))
    print("XGBoost duplicate keys:", int(xk.duplicated().sum()))

    manifest_set = set(map(tuple, mk.astype(str).to_numpy()))
    xgb_set = set(map(tuple, xk.astype(str).to_numpy()))

    print("Manifest keys:", len(manifest_set))
    print("XGBoost keys:", len(xgb_set))
    print("Keys missing from XGBoost:", len(manifest_set - xgb_set))
    print("Extra XGBoost keys:", len(xgb_set - manifest_set))
    print("Exact key-set match:", manifest_set == xgb_set)

    if "true_label" in xgb.columns and "disease_label" in manifest.columns:
        manifest_labels = manifest[key_cols + ["disease_label"]].copy()
        xgb_labels = xgb[key_cols + ["true_label"]].copy()
        joined = manifest_labels.merge(
            xgb_labels, on=key_cols, how="outer", indicator=True
        )
        matched = joined[joined["_merge"] == "both"]
        mismatch_count = int(
            (matched["disease_label"].astype(int)
             != matched["true_label"].astype(int)).sum()
        )
        print("True-label mismatches on matched keys:", mismatch_count)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
NOTEBOOK_DIR = ROOT / "Notebooks 2"

print("========== SVM / RBF ARTIFACT SEARCH ==========")

terms = ("svm", "support_vector", "rbf", "notebook 09", "notebook_09")
matches = []

for folder in [
    ROOT / "data/processed",
    ROOT / "reports",
    ROOT / "models",
    NOTEBOOK_DIR,
]:
    if not folder.exists():
        continue

    for p in folder.rglob("*"):
        if not p.is_file():
            continue
        if any(term in p.name.lower() for term in terms):
            matches.append(p)

for p in sorted(set(matches)):
    print(p.relative_to(ROOT))

print("\n========== NOTEBOOK 09 CONTENT ==========")

notebook_candidates = list(NOTEBOOK_DIR.glob("*09*"))
for nb_path in notebook_candidates:
    if nb_path.suffix != ".ipynb":
        continue

    print("\nNotebook:", nb_path.name)

    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
    except Exception as e:
        print("Could not read notebook:", e)
        continue

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        if any(t in source.lower() for t in ["svm", "rbf", "supportvector", "macro_f1"]):
            print(f"\n--- Cell {i} ({cell.get('cell_type')}) ---")
            print(source[:3500])

            outputs = cell.get("outputs", [])
            for out in outputs:
                txt = "".join(out.get("text", []))
                if txt.strip():
                    print("OUTPUT:", txt[:2000])

print("\n========== SAVED SVM METRIC FILES ==========")

for p in sorted(set(matches)):
    if p.suffix.lower() != ".csv":
        continue
    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    if any("svm" in str(v).lower() or "rbf" in str(v).lower()
           for col in df.columns
           for v in df[col].astype(str).head(100).tolist()):
        print("\nFile:", p.relative_to(ROOT))
        print("Shape:", df.shape)
        print(df.head(10).to_string(index=False))


In [ ]:

from pathlib import Path
import joblib
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

ARTIFACTS = [
    ROOT / "models/traditional_ml/svm.joblib",
    ROOT / "models/traditional_ml_patient_disjoint/svm.joblib",
]

print("=" * 80)
print("SAVED SVM ARTIFACT INSPECTION")
print("=" * 80)

for path in ARTIFACTS:
    print("\n" + "-" * 80)
    print("Artifact:", path.relative_to(ROOT))
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print("File size (bytes):", path.stat().st_size)

    try:
        model = joblib.load(path)
    except Exception as e:
        print("Could not load artifact:", repr(e))
        continue

    print("Loaded object type:", type(model).__name__)
    print("Module:", type(model).__module__)

    # Inspect pipeline steps or the estimator itself.
    if hasattr(model, "named_steps"):
        print("\nPipeline steps:")
        for step_name, estimator in model.named_steps.items():
            print(f"  {step_name}: {type(estimator).__name__}")
            if hasattr(estimator, "get_params"):
                params = estimator.get_params(deep=False)
                for key in [
                    "kernel", "C", "gamma", "class_weight",
                    "probability", "n_estimators", "max_depth",
                    "random_state"
                ]:
                    if key in params:
                        print(f"    {key}: {params[key]}")
    elif hasattr(model, "get_params"):
        params = model.get_params(deep=False)
        print("\nEstimator parameters:")
        for key in [
            "kernel", "C", "gamma", "class_weight",
            "probability", "n_estimators", "max_depth",
            "random_state"
        ]:
            if key in params:
                print(f"  {key}: {params[key]}")

    # Look for common fitted preprocessing/model attributes.
    for attr in [
        "n_features_in_", "classes_", "feature_names_in_",
        "best_params_", "best_score_"
    ]:
        if hasattr(model, attr):
            value = getattr(model, attr)
            try:
                value = value.tolist()
            except Exception:
                pass
            print(f"{attr}: {value}")

# Inspect nearby metadata/feature lists that may identify the artifact protocol.
print("\n" + "=" * 80)
print("RELATED SVM METADATA")
print("=" * 80)

search_roots = [
    ROOT / "models/traditional_ml",
    ROOT / "models/traditional_ml_patient_disjoint",
    ROOT / "data/processed/classical_ml",
    ROOT / "reports/research_audit/common_patient_split",
]

for folder in search_roots:
    if not folder.exists():
        continue

    for p in sorted(folder.iterdir()):
        if p.is_file() and (
            "svm" in p.name.lower()
            or "feature" in p.name.lower()
            or "manifest" in p.name.lower()
        ):
            print(p.relative_to(ROOT))


In [ ]:

from pathlib import Path
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

folders = [
    ROOT / "models/traditional_ml",
    ROOT / "models/traditional_ml_patient_disjoint",
]

feature_lists = {}

for folder in folders:
    print("\n" + "=" * 80)
    print("DIRECTORY:", folder.relative_to(ROOT))

    if not folder.exists():
        print("Directory missing")
        continue

    for p in sorted(folder.iterdir()):
        if p.is_file():
            print(f"{p.name:45s} {p.stat().st_size:>10,} bytes")

    feature_path = folder / "selected_features.json"
    if feature_path.exists():
        try:
            content = json.loads(feature_path.read_text(encoding="utf-8"))
            feature_lists[folder.name] = content
            print("\nFeature metadata:")
            print(json.dumps(content, indent=2)[:5000])
        except Exception as e:
            print("Could not read feature metadata:", repr(e))

print("\n" + "=" * 80)
print("FEATURE-LIST COMPARISON")

if len(feature_lists) == 2:
    a = feature_lists.get("traditional_ml")
    b = feature_lists.get("traditional_ml_patient_disjoint")

    def extract_features(obj):
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            for key in ["selected_features", "feature_names", "features"]:
                if key in obj and isinstance(obj[key], list):
                    return obj[key]
        return None

    fa = extract_features(a)
    fb = extract_features(b)

    if fa is not None and fb is not None:
        print("Traditional ML feature count:", len(fa))
        print("Patient-disjoint feature count:", len(fb))
        print("Same feature order:", fa == fb)
        print("Same features, ignoring order:", set(fa) == set(fb))
        print("Only in traditional ML:", sorted(set(fa) - set(fb)))
        print("Only in patient-disjoint:", sorted(set(fb) - set(fa)))
    else:
        print("Could not automatically extract feature names.")
        print("Inspect the metadata shown above.")
else:
    print("Both feature-list files were not available.")


In [ ]:

from pathlib import Path
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

for rel in [
    "models/traditional_ml/classical_ml_metadata.json",
    "models/traditional_ml_patient_disjoint/classical_ml_metadata.json",
]:
    path = ROOT / rel
    print("\n" + "=" * 90)
    print(rel)

    if path.exists():
        try:
            metadata = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(metadata, indent=2, default=str))
        except Exception as e:
            print("Could not parse metadata:", repr(e))
    else:
        print("Not found")

# Also inspect the label mappings in each saved encoder.
import joblib

for rel in [
    "models/traditional_ml/label_encoder.joblib",
    "models/traditional_ml_patient_disjoint/label_encoder.joblib",
]:
    path = ROOT / rel
    print("\n" + "-" * 90)
    print(rel)

    if path.exists():
        encoder = joblib.load(path)
        print("Classes:", getattr(encoder, "classes_", "No classes_ attribute"))


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

files = [
    "reports/research_audit/common_patient_split/overall_metrics.csv",
    "reports/research_audit/common_patient_split/final_comparison/deduplicated_common_test_leaderboard.csv",
    "reports/research_audit/common_patient_split/test_predictions.csv",
    "reports/research_audit/common_patient_split/confusion_matrix_svm.csv",
]

for rel in files:
    p = ROOT / rel
    print("\n" + "=" * 85)
    print(rel)

    if not p.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(p)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "model_id" in df.columns:
        print("Model IDs:", sorted(df["model_id"].dropna().astype(str).unique()))
        svm_rows = df[df["model_id"].astype(str).str.lower().str.contains("svm")]
        print("SVM rows:")
        print(svm_rows.head(10).to_string(index=False))

    elif "model" in df.columns:
        print("Model values:", df["model"].astype(str).unique().tolist())
        print(df[df["model"].astype(str).str.lower().str.contains("svm")].to_string(index=False))

    else:
        print(df.head(8).to_string(index=False))


In [ ]:

from pathlib import Path

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

search_roots = [
    ROOT / "reports/research_audit",
    ROOT / "data/processed",
]

terms = [
    "test_cnn_manifest.csv",
    "train_cnn_manifest.csv",
    "validation_cnn_manifest.csv",
    "common_patient_split_manifest",
    "test_predictions.csv",
]

for folder in search_roots:
    print(f"\n{'=' * 80}\nSEARCHING: {folder}")
    if not folder.exists():
        print("Folder not found")
        continue

    for p in folder.rglob("*"):
        if p.is_file() and any(term in p.name.lower() for term in terms):
            print(p.relative_to(ROOT))

print("\n" + "=" * 80)
print("Other possible test manifests:")
for folder in search_roots:
    if folder.exists():
        for p in folder.rglob("*.csv"):
            name = p.name.lower()
            if "manifest" in name and ("test" in name or "split" in name):
                print(p.relative_to(ROOT))


In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
MODEL_DIR = ROOT / "models/traditional_ml_patient_disjoint"
FEATURE_PATH = ROOT / "data/processed/features/respiratory_cycle_features.csv"
MANIFEST_PATH = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"

# Load the saved inference artifacts; no training or saving.
svm = joblib.load(MODEL_DIR / "svm.joblib")
scaler = joblib.load(MODEL_DIR / "scaler.joblib")
encoder = joblib.load(MODEL_DIR / "label_encoder.joblib")
features = json.loads(
    (MODEL_DIR / "selected_features.json").read_text(encoding="utf-8")
)

X_all = pd.read_csv(FEATURE_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

print("Manifest shape:", manifest.shape)
print("Manifest columns:", manifest.columns.tolist())
print("Feature table shape:", X_all.shape)

keys = ["patient_id", "recording_id", "chest_location", "cycle_number"]
missing_keys = (
    [c for c in keys if c not in manifest.columns]
    + [c for c in keys if c not in X_all.columns]
)
missing_features = [c for c in features if c not in X_all.columns]

if missing_keys or missing_features:
    raise ValueError(
        f"Missing keys: {missing_keys}; missing selected features: {missing_features}"
    )

if manifest.duplicated(keys).any() or X_all.duplicated(keys).any():
    raise ValueError("Duplicate cycle keys found; stop before evaluation.")

test = manifest[keys].merge(
    X_all[keys + features],
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Join status:", test["_merge"].value_counts().to_dict())
if not (test["_merge"] == "both").all():
    raise ValueError("Some test cycles do not have matching features.")

X_test = test[features].apply(pd.to_numeric, errors="coerce")
if not np.isfinite(X_test.to_numpy(dtype=float)).all():
    raise ValueError("Missing or non-finite values in selected features.")

print("\nManifest sample:")
print(manifest.head(3).to_string(index=False))

# Stop here if the diagnosis column name is not yet known.
label_candidates = [
    c for c in manifest.columns
    if any(term in c.lower() for term in ["disease", "diagnosis", "label", "class"])
]
print("\nPossible label columns:", label_candidates)

# Automatically choose a disease-name column only when it matches encoder classes.
class_names = set(map(str, encoder.classes_))
label_col = None

for c in ["disease", "disease_label", "label", "true_disease",
          "diagnosis", "class_name", "class_label"]:
    if c in manifest.columns:
        values = set(manifest[c].dropna().astype(str).unique())
        if values and values.issubset(class_names):
            label_col = c
            break

if label_col is None:
    raise ValueError(
        "Could not safely identify the diagnosis column. "
        "Share the printed manifest columns and sample rows."
    )

# Inference using the matching saved scaler and SVM.
X_scaled = scaler.transform(X_test)
pred_encoded = svm.predict(X_scaled)
pred_labels = encoder.inverse_transform(np.asarray(pred_encoded, dtype=int))
truth = manifest[label_col].astype(str).to_numpy()

print("\nVerified artifact:", MODEL_DIR / "svm.joblib")
print("Truth column:", label_col)
print("Test cycles:", len(truth))
print("Accuracy:", round(accuracy_score(truth, pred_labels), 6))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(truth, pred_labels), 6),
)
print(
    "Macro-F1:",
    round(f1_score(truth, pred_labels, average="macro", zero_division=0), 6),
)
print(
    "Weighted F1:",
    round(f1_score(truth, pred_labels, average="weighted", zero_division=0), 6),
)
print("\nClassification report:")
print(classification_report(truth, pred_labels, zero_division=0))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

manifest = pd.read_csv(
    ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"
)
features = pd.read_csv(
    ROOT / "data/processed/features/respiratory_cycle_features.csv"
)

print("=" * 75)
print("MANIFEST SAMPLE")
print(manifest.head(8).to_string(index=False))
print("\nManifest dtypes:")
print(manifest.dtypes.to_string())

print("\n" + "=" * 75)
print("FEATURE TABLE SAMPLE")
print(features.head(8).to_string(index=False))
print("\nFeature-table columns:")
print(features.columns.tolist())

# Find likely audio-path and cycle identifiers.
for name, df in [("manifest", manifest), ("features", features)]:
    print(f"\n{name.upper()} possible identifier columns:")
    print([
        c for c in df.columns
        if any(t in c.lower() for t in [
            "audio", "path", "cycle", "recording", "patient",
            "chest", "diagnosis", "disease", "split"
        ])
    ])

# Check whether audio_path can be used to align rows.
if "audio_path" in manifest.columns and "audio_path" in features.columns:
    print("\nAudio-path key checks:")
    print("Manifest duplicate paths:", manifest["audio_path"].duplicated().sum())
    print("Feature duplicate paths:", features["audio_path"].duplicated().sum())

    mpaths = set(manifest["audio_path"].dropna().astype(str))
    fpaths = set(features["audio_path"].dropna().astype(str))
    print("Manifest unique paths:", len(mpaths))
    print("Feature unique paths:", len(fpaths))
    print("Manifest paths absent from features:", len(mpaths - fpaths))
    print("Feature paths absent from manifest:", len(fpaths - mpaths))

# Inspect the existing CNN predictions for alignment clues.
pred_path = (
    ROOT / "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv"
)
print("\n" + "=" * 75)
print("CNN TEST PREDICTIONS")
print("Exists:", pred_path.exists())

if pred_path.exists():
    pred = pd.read_csv(pred_path)
    print("Shape:", pred.shape)
    print("Columns:", pred.columns.tolist())
    print(pred.head(5).to_string(index=False))


In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)
MODEL_DIR = ROOT / "models/traditional_ml_patient_disjoint"
FEATURE_PATH = ROOT / "data/processed/features/respiratory_cycle_features.csv"
MANIFEST_PATH = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"

# Load saved artifacts; no training and no files are modified.
svm = joblib.load(MODEL_DIR / "svm.joblib")
scaler = joblib.load(MODEL_DIR / "scaler.joblib")
encoder = joblib.load(MODEL_DIR / "label_encoder.joblib")
selected_features = json.loads(
    (MODEL_DIR / "selected_features.json").read_text(encoding="utf-8")
)

manifest = pd.read_csv(MANIFEST_PATH)
feature_table = pd.read_csv(FEATURE_PATH)

def normalize_cycle_path(value):
    """Normalize absolute and relative cycle paths to data/processed/audio/cycles/..."""
    value = str(value).replace("\\", "/")
    marker = "data/processed/audio/cycles/"
    if marker in value:
        return value[value.index(marker):]
    marker = "audio/cycles/"
    if marker in value:
        return "data/processed/" + value[value.index(marker):]
    raise ValueError(f"Unrecognized cycle path: {value}")

manifest["cycle_key"] = manifest["audio_path"].map(normalize_cycle_path)
feature_table["cycle_key"] = feature_table["processed_path"].map(
    normalize_cycle_path
)

print("Test manifest rows:", len(manifest))
print("Feature table rows:", len(feature_table))
print("Duplicate manifest cycle keys:", manifest["cycle_key"].duplicated().sum())
print("Duplicate feature cycle keys:", feature_table["cycle_key"].duplicated().sum())

if manifest["cycle_key"].duplicated().any():
    raise ValueError("Manifest contains duplicate cycle paths.")
if feature_table["cycle_key"].duplicated().any():
    raise ValueError("Feature table contains duplicate cycle paths.")

manifest_keys = set(manifest["cycle_key"])
feature_keys = set(feature_table["cycle_key"])
print("Test cycles without features:", len(manifest_keys - feature_keys))
print("Feature cycles outside test set:", len(feature_keys - manifest_keys))

# Match each manifest cycle to exactly one feature row.
joined = manifest.merge(
    feature_table[["cycle_key"] + selected_features],
    on="cycle_key",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Join status:", joined["_merge"].value_counts().to_dict())
if not (joined["_merge"] == "both").all():
    raise ValueError("Not every test cycle matched exactly one feature row.")

X_test = joined[selected_features].apply(pd.to_numeric, errors="coerce")
if not np.isfinite(X_test.to_numpy(dtype=float)).all():
    raise ValueError("Selected features contain missing or non-finite values.")

# Inference: saved scaler -> saved RBF SVM -> saved label encoder.
X_scaled = scaler.transform(X_test)
pred_encoded = svm.predict(X_scaled)
predicted = encoder.inverse_transform(np.asarray(pred_encoded, dtype=int))

# Verify the feature table's diagnosis agrees with the manifest.
# The diagnosis column is not used as an input feature.
diag_check = manifest[["cycle_key", "diagnosis"]].merge(
    feature_table[["cycle_key", "diagnosis"]],
    on="cycle_key",
    how="left",
    suffixes=("_manifest", "_features"),
    validate="one_to_one",
)
diagnosis_mismatches = (
    diag_check["diagnosis_manifest"].astype(str)
    != diag_check["diagnosis_features"].astype(str)
).sum()
print("Diagnosis mismatches between manifest and feature table:", diagnosis_mismatches)

truth = joined["diagnosis"].astype(str).to_numpy()
supported = sorted(set(truth))

print("\n========== SAVED PATIENT-DISJOINT SVM ==========")
print("Kernel:", svm.kernel)
print("C:", svm.C, "| gamma:", svm.gamma, "| class_weight:", svm.class_weight)
print("Test cycles:", len(truth))
print("Test patients:", joined["patient_id"].nunique())
print("Supported test classes:", supported)
print("Accuracy:", round(accuracy_score(truth, predicted), 6))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(truth, predicted), 6),
)
print(
    "Macro-F1 (supported classes):",
    round(f1_score(truth, predicted, labels=supported,
                   average="macro", zero_division=0), 6),
)
print(
    "Macro-F1 (all 8 classes):",
    round(f1_score(truth, predicted, labels=list(encoder.classes_),
                   average="macro", zero_division=0), 6),
)
print(
    "Weighted F1:",
    round(f1_score(truth, predicted, average="weighted", zero_division=0), 6),
)
print("\nClassification report:")
print(classification_report(truth, predicted, labels=list(encoder.classes_),
                            zero_division=0))
